# 6 自定义量化 A8W8 matmul 算子开发并接入 Qwen3-8B

本章是整个启航营的**核心实践环节**，将学习到的所有知识沉淀转化为一次完整的自定义算子开发实战。

实践的环境准备、提交方式可参考[启航营课程实践说明](https://gitcode.com/cann/cann-launch-camp/blob/master/2026/University/HIT/First-session/README.md)。

## 本章定位

本章将开发一个自定义的量化 matmul 算子 `QmmCustom`，通过 A8W8 量化（Activation INT8 + Weight INT8）减少计算量和内存占用，从而提升推理性能。这是本次启航营中综合性最强的一章——涵盖了从算子概念理解、Tiling 设计、Kernel 实现到算子编译、接入网络与功能性能测试的全链路。

## 通过实践达成的学习目标

- **理解量化推理原理**：掌握 A8W8 量化方案（INT8 激活 + INT8 权重）的数学原理与 Scale 反量化机制，理解 perChannelScale 与 perTokenScale 的作用。
- **设计 Tiling 数据结构与函数**：根据算子原型定义自行设计 TilingData 结构体，实现多核分块的 Tiling 函数，理解 Cube 计算单元的分块策略。
- **实现 Cube+Vector 双路径 Kernel**：完成 Cube-only（INT32 输出）和 Cube+Vector 协作（反量化 BF16 输出）两条 Kernel 路径的实现，理解昇腾 AI Core 的 Cube 与 Vector 协作编程模型。
- **完成算子编译与 Torch 接入**：编译自定义算子并通过 Torch 接口接入，打通“算子开发 → 框架调用”的链路。
- **测试算子功能与性能，尝试优化迭代**：验证自定义算子的精度正确性，测试单算子与网络性能，并根据测试结果对 Tiling 或 Kernel 进行优化迭代。

## 本章大纲

- 环境准备
- A8W8 量化 matmul 算子概念
- 自定义算子实现
    - **任务一：自行设计tilingdata结构体并实现tiling函数**
    - **任务二：实现A8W8输入，INT32输出的kernel**
    - **任务三：实现A8W8输入，经过perChannelScale和perTokenScale反量化输出BF16的kernel**
- 编译自定义算子
    - **任务四：成功编译自定义算子**
- 测试自定义算子功能性能
    - **任务五：根据提供的测试脚本，测试自定义算子功能与性能，尝试对自定义算子进行优化后重复整个流程**
- 自定义算子接入模型
- 模型性能测试
    - **任务六：测试模型性能，尝试对自定义算子进行优化后重复上述流程**
- 提交要求
---

## 1 环境准备

定位教程目录、准备运行目录，并把 CANN 环境变量导入当前 Notebook 进程：

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

print('[环境准备] 开始', flush=True)

def locate_repo_root():
    repo_roots = []
    try:
        cwd = Path.cwd()
        lineage = [cwd, *cwd.parents]
        repo_roots.extend(lineage)
        repo_roots.extend(base / 'cann-learning-hub' for base in lineage)
        for base in lineage:
            try:
                repo_roots.extend(base.glob('*/cann-learning-hub'))
            except OSError:
                pass
    except FileNotFoundError:
        pass
    repo_roots.append(Path('/opt/atomgit/cann-learning-hub'))

    seen = set()
    for repo_root in repo_roots:
        key = str(repo_root)
        if key in seen:
            continue
        seen.add(key)
        if (repo_root / 'tutorials/llm_inference/qwen3_8b/src').exists():
            return repo_root
    raise FileNotFoundError('Cannot locate cann-learning-hub repository root.')

REPO_ROOT = locate_repo_root()
os.chdir(REPO_ROOT)
TUTORIAL_DIR = REPO_ROOT / 'tutorials' / 'llm_inference' / 'qwen3_8b'
TUTORIAL_SRC_DIR = TUTORIAL_DIR / 'src'
if str(TUTORIAL_SRC_DIR) not in sys.path:
    sys.path.insert(0, str(TUTORIAL_SRC_DIR))
from inference_scripts.environment_check import require_unified_environment
require_unified_environment(TUTORIAL_DIR / 'requirements.txt')
SRC_DIR = TUTORIAL_DIR / 'src' / 'op_custom'

if not os.environ.get('ASCEND_TOOLKIT_HOME'):
    raise EnvironmentError('ASCEND_TOOLKIT_HOME is not set. Please activate the CANN environment.')
print('CANN包路径 =', os.environ.get('ASCEND_TOOLKIT_HOME'))
cann_script = Path(os.environ['ASCEND_TOOLKIT_HOME']) / 'set_env.sh'
env_cmd = f'source {cann_script} && env'
env = subprocess.check_output(f"bash -lc '{env_cmd}'", shell=True, text=True, cwd=REPO_ROOT)
for line in env.splitlines():
    if '=' in line:
        os.environ.__setitem__(*line.split('=', 1))

RECIPE_ROOT = TUTORIAL_DIR / 'src/inference_scripts/recipe_qwen3_8b'
QMM_CUSTOM_DIR = SRC_DIR / 'qmm_custom'

print('教程目录 =', TUTORIAL_DIR)
print('推理模型目录 =', RECIPE_ROOT)
print('算子相关文件目录 =', SRC_DIR)
print('算子实现目录 =', QMM_CUSTOM_DIR)
print('[环境准备] 完成', flush=True)

[环境准备] 开始


[依赖检查] 统一环境已就绪，复用第 2 章环境
CANN包路径 = /home/developer/Ascend/cann-9.0.0
教程目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b
推理模型目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b
算子相关文件目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom
算子实现目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom
[环境准备] 完成


本章直接复用第 2 章检查完成的统一环境，不再执行依赖安装。

---
## 2 A8W8 量化 matmul 算子概念

### 2.1 什么是 A8W8 量化

A8W8 量化是 LLM 推理中常用的一种量化方案：
- **A（Activation）**：将输入激活值从 BF16/FP16 量化为 **INT8**
- **W（Weight）**：将权重从 BF16/FP16 量化为 **INT8**
- **反量化**：计算结果 `INT8 × INT8 → INT32` 后，通过 Scale 反量化回浮点精度

> **术语说明**：A8W8 与第 5 章使用的 W8A8 是同一种 INT8 激活 + INT8 权重量化方案的不同写法，本章沿用 A8W8 命名。仓库中 YAML 配置文件名（如 `qwen3_8b_a8w8_1tp.yaml`）和类名（`CompressedTensorsW8A8Int8LinearMethod`）也分别使用了这两种写法。

### 2.2 算子规格

在上一章中，通过分析量化模型的profiling，归纳出了自定义算子的输入输出规格：

| 算子名称 | QmmCustom |
|------|-----|
| 输入 x1 (A) | INT8, ND 格式, shape [M, K] |
| 输入 x2 (B) | INT8, **FRACTAL_NZ** 格式, shape [K, N] |
| 输入 scale | FLOAT32, ND 格式, shape [N] |
| 输入 pertoken_scale | FLOAT32, ND 格式, shape [M]（可选） |
| 输出 y | 有 perTokenScale → BF16，无 → INT32；ND 格式, shape [M, N] |
| 架构 | ascend910b |

**注意**：x2 权重矩阵使用 **FRACTAL_NZ** 格式，这是昇腾 AI Core Cube 单元的高效存储格式，能提升 matmul 计算性能。

### 2.3 Scale 模式

QmmCustom 算子支持两种 Scale：

| Scale 类型 | Shape | 作用 | 
|-----------|-------|------|
| **perChannelScale** | `[N]` | 对每个输出通道单独缩放 |
| **perTokenScale** | `[M]` | 对每行输入单独缩放 |

计算公式：
```
result_bf16 = (x1_int8 @ x2_int8) * perChannelScale * perTokenScale
```

## 3 自定义算子实现

与前面学习过的内容一致，本次实践采用 `<<<>>>` Kernel 直调方式开发上述原型的量化 matmul 算子。直调方式通过 `kernel<<<numBlocks, dynUBufSize, stream>>>(args)` 语法在 Host 侧直接启动 Kernel，适合快速验证算子功能与性能。

本节包含如下三个任务：

### 任务一：自行设计 TilingData 结构体并实现 Tiling 函数

根据算子原型与 Kernel 的实现需求，自行设计 `QmmCustomTilingData` 结构体，并实现 `CalcQmmTiling` 函数。Tiling 函数需根据输入的 M、N、K 维度，结合平台硬件资源（L1/L0A/L0B/UB 内存限制、可用核数），计算多核分块参数。可借助 `platform_ascendc::PlatformAscendCManager` 获取硬件信息。

> **Tiling 设计提示**：
> 
> 设计 TilingData 时需要考虑以下要素：
> 1. **`TCubeTiling`**：CANN Matmul 库提供的标准 Tiling 结构体，包含 M/N/K 的分块大小（`baseM`、`baseN`、`baseK`）、使用的核数（`usedCoreNum`）等字段。可通过 `GetMatmulTiling` API 自动计算。
> 2. **`isPertoken`**：标记是否走反量化路径，用于 Kernel 内部分支选择。
> 3. **`workspaceSize`**：Path 2 中 Cube 写入 INT32 中间结果所需的 GM workspace 大小。
> 4. **多核分块策略**：将 M 维度或 N 维度分配到多个核上并行计算，每个核处理一个分块。分块大小需平衡 L1/L0A/L0B 缓存利用率与多核负载均衡。

### 任务二：实现 A8W8 输入、INT32 输出的 Kernel

实现 `QmmCubeBasicKernel` 类，完成纯 Cube 路径的 INT8 矩阵乘法计算。

计算公式： ```result_int32 = x1_int8 @ x2_int8```

### 任务三：实现 A8W8 输入、经 perChannelScale 和 perTokenScale 反量化输出 BF16 的 Kernel

实现 `QmmPertokenKernel` 类，在 Cube 计算的基础上增加 Vector 反量化路径。

计算公式： ```result_bf16 = (x1_int8 @ x2_int8) * perChannelScale * perTokenScale```

其中 `perChannelScale` 沿 N 维广播，`perTokenScale` 沿 M 维广播，两者均为 FLOAT32 类型。

---

### 执行路径说明

在 Host 侧的 `qmm_custom` 函数中，根据是否传入 `pertoken_scale` 设置 `isPertoken` 标志，Kernel 启动时据此选择执行路径：

| 路径 | 条件 | 计算流程 | 输出类型 |
|------|------|----------|----------|
| **Path 1: Cube-only** | `isPertoken == 0` | INT8 matmul → INT32 | INT32 |
| **Path 2: Cube+Vector** | `isPertoken == 1` | INT8 matmul → INT32 → dequant(scale×pertoken) → BF16 | BF16 |

**Path 1（Cube-only）**：仅使用 AI Core 的 Cube 计算单元，完成 INT8×INT8 矩阵乘法后直接将 INT32 结果写入输出 GM，无 Vector 参与。对应 `QmmCubeBasicKernel` 的实现。

**Path 2（Cube+Vector）**：Cube 单元先完成 INT8 matmul 得到 INT32 中间结果并写入 GM workspace；随后 Vector 单元从 GM 读取 INT32 结果，依次完成 `Cast`（INT32→FLOAT32）、乘以 `perChannelScale` 和 `perTokenScale`、`Cast`（FLOAT32→BF16）等反量化操作，最终将 BF16 结果写入输出 GM。对应 `QmmPertokenKernel` 的实现。

Kernel 入口函数 `qmm_custom_kernel` 使用 `__mix__(1, 2)` 属性声明，表示为每个 Block 分配 1 个 Cube 核和 2 个 Vector 核，使 Cube 与 Vector 可在同一 Kernel 中协同工作。Kernel 内部通过 `tilingData.isPertoken` 判断走哪条路径，两条路径复用相同的 Tiling 结构和 Kernel 启动入口。

Tiling、Kernel、Torch 接口的实现均位于 `qmm_custom.asc` 文件中。文件已给出整体框架与函数签名，请自行完成下方代码中的 TODO 部分，以实现完整的自定义算子：

In [2]:
%%writefile {QMM_CUSTOM_DIR}/qmm_custom.asc
#include "acl/acl_base.h"
#include "acl/acl_prof.h"
#include "kernel_operator.h"
#define ASCENDC_CUBE_ONLY                       // must precede lib/matmul_intf.h
#include "lib/matmul_intf.h"
#include "tiling/platform/platform_ascendc.h"
#include "tiling/tiling_api.h"
#include "torch_npu/csrc/core/npu/NPUStream.h"
#include <torch/extension.h>
#include <cstdint>
#include "kernel_tiling/kernel_tiling.h"
#include "utils/std/cmath.h"
using namespace AscendC;

// ---------------------------------------------------------------------------
// Constants (ported from template kernel.asc:459-480)
// ---------------------------------------------------------------------------
constexpr bool    IS_TRANS_A = false;
constexpr bool    IS_TRANS_B = false;
constexpr bool    IS_BIAS    = false;
constexpr int64_t QMM_K0 = 16;
constexpr int64_t QMM_N0 = 32;
constexpr uint32_t QMM_DEQ_CHUNK_N = 8192;
constexpr uint16_t QMM_AIC2AIV_READY = 5;      // AIC->AIV per-tile ready
constexpr uint16_t QMM_AIV2AIC_RELEASE = 3;    // AIV->AIC slot release
constexpr int32_t  QMM_RING_DEPTH = 4;         // <=8 keeps pending flag <=15

// ===========================================================================
// L2CacheOptimizer (template kernel.asc:39-243)
// ===========================================================================
namespace CustomMatmulL2Cache {
constexpr int64_t L2_TILE_THRESHOLD = 64 * 1024 * 1024;
constexpr int64_t L1_MIN_UST_DIM = 4;
constexpr int64_t L1_MAX_UST_DIM = 8;

template <size_t I, typename T>
__aicore__ constexpr inline decltype(auto) Get(T&& t)
{
    return AscendC::Std::get<I>(AscendC::Std::forward<T>(t));
}

template <class ProblemShape, const auto& MmTiling>
class L2CacheOptimizer {
public:
    using BlockShape = AscendC::Std::tuple<int64_t, int64_t, int64_t>;
    using BlockCoord = AscendC::Std::tuple<int64_t, int64_t, int64_t>;
    static constexpr int64_t l1M = MmTiling.baseM * MmTiling.stepM;
    static constexpr int64_t l1N = MmTiling.baseN * MmTiling.stepN;
    static constexpr int64_t l0M = MmTiling.baseM;
    static constexpr int64_t l0N = MmTiling.baseN;
    static constexpr int64_t l0K = MmTiling.baseK;

private:
    int64_t mTileNum_, nTileNum_, blockNum_, m_, n_, k_, totalTileNum_;
    int64_t newBlockIdx_, mL2TileNumTmp_, nL2TileNumTmp_, nL2Idx_, mL2Idx_;
    int64_t mL2BlockNum_, nL2BlockNum_, mL2TileNum_, nL2TileNum_;
public:
    __aicore__ inline L2CacheOptimizer(ProblemShape shape, int64_t blockNum) : blockNum_(blockNum)
    {
        m_ = shape.m; n_ = shape.n; k_ = shape.k;
        mTileNum_ = AscendC::Std::ceil_division(m_, l1M);
        nTileNum_ = AscendC::Std::ceil_division(n_, l1N);
        totalTileNum_ = mTileNum_ * nTileNum_;
        InitL2Tile();
    }

    __aicore__ inline int64_t GetTileNum() { return totalTileNum_; }

    __aicore__ inline BlockShape GetBlockShape(int64_t tileIdx)
    {
        GetCommonTileIndex(tileIdx);
        int64_t mTileIdx = newBlockIdx_ % mL2TileNumTmp_;
        mTileIdx = mTileIdx + mL2Idx_ * mL2TileNum_;
        int64_t nTileIdx = 0;
        if (mL2TileNumTmp_ != 0 && nL2TileNumTmp_ != 0) {
            int64_t tmp = newBlockIdx_ / CalcLcm(mL2TileNumTmp_, nL2TileNumTmp_);
            nTileIdx = (newBlockIdx_ + tmp) % nL2TileNumTmp_;
        }
        nTileIdx = nTileIdx + nL2Idx_ * nL2TileNum_;
        int64_t tailL1M = (m_ % l1M == 0) ? l1M : m_ % l1M;
        int64_t tailL1N = (n_ % l1N == 0) ? l1N : n_ % l1N;
        int64_t blockShapeM = IsMTail(mTileIdx, mTileNum_) ? tailL1M : l1M;
        int64_t blockShapeN = IsNTail(nTileIdx, nTileNum_) ? tailL1N : l1N;
        return {blockShapeM, blockShapeN, k_};
    }

    __aicore__ inline BlockCoord GetBlockCoord(int64_t tileIdx)
    {
        GetCommonTileIndex(tileIdx);
        int64_t mTileIdx = newBlockIdx_ % mL2TileNumTmp_;
        mTileIdx = mTileIdx + mL2Idx_ * mL2TileNum_;
        int64_t nTileIdx = 0;
        if (mL2TileNumTmp_ != 0 && nL2TileNumTmp_ != 0) {
            int64_t tmp = newBlockIdx_ / CalcLcm(mL2TileNumTmp_, nL2TileNumTmp_);
            nTileIdx = (newBlockIdx_ + tmp) % nL2TileNumTmp_;
        }
        nTileIdx = nTileIdx + nL2Idx_ * nL2TileNum_;
        return {mTileIdx * l1M, nTileIdx * l1N, 0};
    }

private:
    // A int8 (1B), B int8 (1B), C int32 (4B)
    __aicore__ inline int64_t GetTotalSize(int64_t mL2, int64_t nL2, int64_t kL2)
    {
        int64_t sizeA = mL2 * kL2;        // int8
        int64_t sizeB = kL2 * nL2;        // int8
        int64_t sizeC = mL2 * nL2 * 4;    // int32
        return sizeA + sizeB + sizeC;
    }
    __aicore__ inline bool EnableL2Tile() { return GetTotalSize(m_, n_, k_) > L2_TILE_THRESHOLD; }
    __aicore__ inline int64_t GetTail(int64_t dividend, int64_t divisor)
    {
        if (divisor == 0) return 0;
        auto remainder = dividend % divisor;
        return (remainder == 0) ? divisor : remainder;
    }
    __aicore__ inline void InitL2TileTail()
    {
        int64_t mConflict = INT64_MAX, nConflict = INT64_MAX;
        constexpr bool isNMajor = l1N > l1M;
        int64_t majorDim = isNMajor ? nTileNum_ : mTileNum_;
        int64_t minorDim = isNMajor ? mTileNum_ : nTileNum_;
        int64_t maxMajor = blockNum_ > majorDim ? majorDim : blockNum_;
        int64_t maxMinor = blockNum_ > minorDim ? minorDim : blockNum_;
        int64_t minMinor = isNMajor ? L1_MIN_UST_DIM : L1_MAX_UST_DIM;
        for (int64_t i = maxMajor; i >= L1_MIN_UST_DIM; i--) {
            for (int64_t j = maxMinor; j >= minMinor; j--) {
                if (GetTotalSize(j * l1M, i * l1N, k_) > L2_TILE_THRESHOLD) continue;
                int64_t curMajorDim = isNMajor ? j : i;
                int64_t curMinorDim = isNMajor ? i : j;
                int64_t mL2TileNumTailTmp = GetTail(mTileNum_, curMajorDim);
                int64_t nL2TileNumTailTmp = GetTail(nTileNum_, curMinorDim);
                uint64_t mConflictTmp = AscendC::Std::ceil_division(blockNum_, mL2TileNumTailTmp);
                uint64_t nConflictTmp = AscendC::Std::ceil_division(blockNum_, nL2TileNumTailTmp);
                if (mConflict >= mConflictTmp && nConflict >= nConflictTmp) {
                    mConflict = mConflictTmp; nConflict = nConflictTmp;
                    mL2TileNum_ = curMajorDim; nL2TileNum_ = curMinorDim;
                }
            }
        }
        if (mL2TileNum_ == 0 || nL2TileNum_ == 0) { mL2TileNum_ = mTileNum_; nL2TileNum_ = nTileNum_; }
    }
    __aicore__ inline void InitL2Tile()
    {
        bool smallDim = mTileNum_ < L1_MIN_UST_DIM && nTileNum_ < L1_MIN_UST_DIM;
        if (smallDim || (!EnableL2Tile())) {
            mL2TileNum_ = mTileNum_; nL2TileNum_ = nTileNum_;
            mL2BlockNum_ = 1; nL2BlockNum_ = 1; return;
        }
        mL2TileNum_ = 0; nL2TileNum_ = 0;
        InitL2TileTail();
        mL2BlockNum_ = AscendC::Std::ceil_division(mTileNum_, mL2TileNum_);
        nL2BlockNum_ = AscendC::Std::ceil_division(nTileNum_, nL2TileNum_);
    }
    __aicore__ inline void GetCommonTileIndex(int64_t tileIdx)
    {
        mL2Idx_ = tileIdx / (mL2TileNum_ * nTileNum_);
        mL2TileNumTmp_ = (mL2Idx_ == mL2BlockNum_ - 1) ? GetTail(mTileNum_, mL2TileNum_) : mL2TileNum_;
        nL2Idx_ = (tileIdx % (mL2TileNum_ * nTileNum_)) / (mL2TileNumTmp_ * nL2TileNum_);
        nL2TileNumTmp_ = (nL2Idx_ == nL2BlockNum_ - 1) ? GetTail(nTileNum_, nL2TileNum_) : nL2TileNum_;
        int64_t startIdx = mL2Idx_ * mL2TileNum_ * nTileNum_ + nL2Idx_ * nL2TileNum_ * mL2TileNumTmp_;
        int64_t startBlockIdx = startIdx % blockNum_;
        newBlockIdx_ = tileIdx - startIdx;
    }
    __aicore__ inline int64_t CalcLcm(int64_t a, int64_t b)
    {
        if (a == 0 || b == 0) return 0;
        int64_t m = a, n = b;
        while (n != 0) { int64_t tmp = m % n; m = n; n = tmp; }
        return (a / m) * b;
    }
    __aicore__ inline bool IsMTail(int64_t mTileIdx, int64_t mTileNum)
    {
        return (mTileIdx - (mTileNum - 1)) % mTileNum == 0;
    }
    __aicore__ inline bool IsNTail(int64_t nTileIdx, int64_t nTileNum) { return nTileIdx == (nTileNum - 1); }
};
} // namespace CustomMatmulL2Cache

namespace qmm_l2c {
struct K7Shape { int32_t m; int32_t n; int32_t k; int32_t isBias; };

__aicore__ inline constexpr MatmulConfig MakeK7Cfg()
{
    MatmulConfig cfg = CFG_MDL;
    cfg.singleCoreM = 30000; cfg.singleCoreN = 30000; cfg.singleCoreK = 30000;
    cfg.basicM = 128; cfg.basicN = 256; cfg.basicK = 128;
    cfg.enableSetBias = false;
    return cfg;
}
__aicore__ inline constexpr MatmulApiStaticTiling GetK7L1Tiling(const MatmulApiStaticTiling& t)
{
    MatmulApiStaticTiling r = t;
    r.stepM = 1; r.stepN = 1; r.stepKa = 4; r.stepKb = 4; r.depthA1 = 8; r.depthB1 = 8;
    return r;
}

template <typename AType, typename BType, typename CType, typename BiasType>
class QmmL2cKernel {
public:
    __aicore__ inline QmmL2cKernel() {}
    using A_TYPE = AscendC::MatmulType<AscendC::TPosition::GM, CubeFormat::ND, AType>;
    using B_TYPE = AscendC::MatmulType<AscendC::TPosition::GM, CubeFormat::NZ, BType>;
    using C_TYPE = AscendC::MatmulType<AscendC::TPosition::GM, CubeFormat::ND, CType>;
    using BIAS_TYPE = AscendC::MatmulType<AscendC::TPosition::GM, CubeFormat::ND, BiasType>;

    constexpr static auto K7_CFG = MakeK7Cfg();
    constexpr static auto K7_CONST_TILING =
        AscendC::GetMatmulApiTiling<A_TYPE, B_TYPE, C_TYPE, BIAS_TYPE>(K7_CFG);
    constexpr static auto K7_MM_TILING = GetK7L1Tiling(K7_CONST_TILING);

    AscendC::Matmul<A_TYPE, B_TYPE, C_TYPE, BIAS_TYPE, K7_MM_TILING> matmulObj;

    __aicore__ inline void Init(__gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* c,
                                int32_t m, int32_t n, int32_t k)
    {
        shapes_.m = m; shapes_.n = n; shapes_.k = k; shapes_.isBias = 0;
        aGlobal_.SetGlobalBuffer(reinterpret_cast<__gm__ AType*>(a), (int64_t)shapes_.m * shapes_.k);
        bGlobal_.SetGlobalBuffer(reinterpret_cast<__gm__ BType*>(b), (int64_t)shapes_.k * shapes_.n);
        cGlobal_.SetGlobalBuffer(reinterpret_cast<__gm__ CType*>(c), (int64_t)shapes_.m * shapes_.n);
    }
    __aicore__ inline void Process()
    {
        const int64_t curBlockIdx = static_cast<int64_t>(AscendC::GetBlockIdx());
        const int64_t blockNum = static_cast<int64_t>(AscendC::GetBlockNum());
        if (curBlockIdx >= blockNum) return;
        CustomMatmulL2Cache::L2CacheOptimizer<K7Shape, K7_MM_TILING> l2Opt(shapes_, blockNum);
        matmulObj.SetOrgShape(shapes_.m, shapes_.n, shapes_.k);
        for (int64_t tileIdx = curBlockIdx; tileIdx < l2Opt.GetTileNum(); tileIdx += blockNum) {
            auto blockShape = l2Opt.GetBlockShape(tileIdx);
            if (CustomMatmulL2Cache::Get<0>(blockShape) <= 0 || CustomMatmulL2Cache::Get<1>(blockShape) <= 0) return;
            auto blockCoord = l2Opt.GetBlockCoord(tileIdx);
            const int64_t mOff = CustomMatmulL2Cache::Get<0>(blockCoord);
            const int64_t nOff = CustomMatmulL2Cache::Get<1>(blockCoord);
            const int64_t offsetA = mOff * shapes_.k;
            const int64_t offsetB = nOff * shapes_.k;
            const int64_t offsetC = mOff * shapes_.n + nOff;
            matmulObj.SetTail((int32_t)CustomMatmulL2Cache::Get<0>(blockShape),
                              (int32_t)CustomMatmulL2Cache::Get<1>(blockShape),
                              (int32_t)CustomMatmulL2Cache::Get<2>(blockShape));
            matmulObj.SetTensorA(aGlobal_[offsetA], false);
            matmulObj.SetTensorB(bGlobal_[offsetB], false);
            matmulObj.IterateAll(cGlobal_[offsetC]);
        }
        matmulObj.End();
    }
private:
    K7Shape shapes_;
    AscendC::GlobalTensor<AType> aGlobal_;
    AscendC::GlobalTensor<BType> bGlobal_;
    AscendC::GlobalTensor<CType> cGlobal_;
};
} // namespace qmm_l2c

// ===========================================================================
// pertoken constant tiling geometry (template kernel.asc:490-516)
// ===========================================================================
namespace qmm_pt_const {
__aicore__ inline constexpr MatmulConfig MakeCfg()
{
    MatmulConfig cfg = CFG_MDL;
    cfg.singleCoreM = 30000; cfg.singleCoreN = 30000; cfg.singleCoreK = 30000;
    cfg.basicM = 128; cfg.basicN = 256; cfg.basicK = 128;
    cfg.enableSetBias = false;
    return cfg;
}
__aicore__ inline constexpr MatmulApiStaticTiling MakeL1Tiling(const MatmulApiStaticTiling& t)
{
    MatmulApiStaticTiling r = t;
    r.stepM = 1; r.stepN = 1; r.stepKa = 4; r.stepKb = 4; r.depthA1 = 8; r.depthB1 = 8;
    r.iterateOrder = 0;
    return r;
}
} // namespace qmm_pt_const

// ===========================================================================
// Optimization switches (compile-time rollback) + no-pt L2C gate
//   S0 baseline: all three optimizations off, sys workspace binding on.
// ===========================================================================
namespace qmm_cfg {
constexpr bool kUseShapeDimOverride = true;    // O5: N==6144 && M<=50 -> dim16
constexpr bool kUseL2C              = true;    // O4: no-pt M>=512 L2 block schedule
constexpr bool kUsePtConst          = true;    // O3: pertoken AIC constant tiling
constexpr bool kSetSysWorkspace     = true;    // workspace sys prefix + SetSysWorkspace
constexpr bool kForceSysWorkspaceFallback = true;  // Force-bind when KFC ptr is null
}
__aicore__ inline bool QmmL2CShapeHit(int64_t m, int64_t k, int64_t n)
{
    if (m < 512 || m > 30000 || n > 30000 || k > 30000) return false;
    return true;
}

// ===========================================================================
// TilingData (scaffold TODO, task 1)
// ===========================================================================
#pragma pack(push, 8)
struct alignas(8) QmmCustomTilingData {
    TCubeTiling cubeTilingData;
    uint32_t isPertoken;
    uint32_t workspaceSize;      // sysWorkspaceSize + ringBytes
    uint32_t sysWorkspaceSize;   // KFC system workspace prefix (512 aligned)
    uint32_t M;
    uint32_t N;
    uint32_t K;
    uint32_t ringDepth;          // QMM_RING_DEPTH, shared by AIC/AIV
    uint32_t reserved;
};
#pragma pack(pop)
static_assert(sizeof(QmmCustomTilingData) % alignof(uint64_t) == 0, "tiling data must stay 8-byte aligned");

// ===========================================================================
// CalcQmmTiling (scaffold TODO, task 1; host side)
// ===========================================================================
static QmmCustomTilingData CalcQmmTiling(uint32_t isPertoken, uint32_t M, uint32_t N, uint32_t K)
{
    QmmCustomTilingData tilingData = {};
    auto ascendcPlatform = platform_ascendc::PlatformAscendCManager::GetInstance();
    TORCH_CHECK(ascendcPlatform != nullptr, "qmm_custom: platform unavailable");

    int32_t dim = static_cast<int32_t>(ascendcPlatform->GetCoreNumAic());
    if (dim < 1) { dim = 1; }

    // O5: restricted shape->dim override (does not change the numeric path)
    if (qmm_cfg::kUseShapeDimOverride && dim > 16 && M <= 50 && N == 6144) { dim = 16; }

    matmul_tiling::MultiCoreMatmulTiling tilingApi(*ascendcPlatform);
    tilingApi.SetDim(dim);
    tilingApi.SetAType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::ND,
                       matmul_tiling::DataType::DT_INT8, IS_TRANS_A);
    tilingApi.SetBType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::NZ,
                       matmul_tiling::DataType::DT_INT8, IS_TRANS_B);
    tilingApi.SetCType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::ND,
                       matmul_tiling::DataType::DT_INT32);
    tilingApi.SetOrgShape(static_cast<int32_t>(M), static_cast<int32_t>(N), static_cast<int32_t>(K));
    tilingApi.SetShape(static_cast<int32_t>(M), static_cast<int32_t>(N), static_cast<int32_t>(K));
    tilingApi.EnableBias(IS_BIAS);
    tilingApi.SetBufferSpace(-1, -1, -1);

    int64_t res = tilingApi.GetTiling(tilingData.cubeTilingData);
    // NZ B slice alignment guard (template kernel.asc:1222-1234)
    if (res == 0 && tilingData.cubeTilingData.usedCoreNum > 0 &&
        (tilingData.cubeTilingData.singleCoreN % QMM_N0) != 0) {
        int32_t alignedSingleN = static_cast<int32_t>((N + dim - 1) / dim);
        alignedSingleN = static_cast<int32_t>(((alignedSingleN + QMM_N0 - 1) / QMM_N0) * QMM_N0);
        if (alignedSingleN > static_cast<int32_t>(N)) alignedSingleN = static_cast<int32_t>(N);
        if (alignedSingleN < QMM_N0) alignedSingleN = static_cast<int32_t>(QMM_N0);
        tilingApi.SetSingleShape(-1, alignedSingleN, -1);
        res = tilingApi.GetTiling(tilingData.cubeTilingData);
    }
    TORCH_CHECK(res != -1, "qmm_custom: GetTiling failed M=", M, " N=", N, " K=", K);

    uint32_t usedCoreNum = static_cast<uint32_t>(tilingData.cubeTilingData.usedCoreNum);
    if (usedCoreNum < 1) { usedCoreNum = 1; }
    tilingData.cubeTilingData.usedCoreNum = static_cast<int32_t>(usedCoreNum);

    tilingData.isPertoken = isPertoken;
    tilingData.M = M; tilingData.N = N; tilingData.K = K;
    tilingData.ringDepth = static_cast<uint32_t>(QMM_RING_DEPTH);

    // System workspace: host must allocate sys + user. On 2201, GetUserWorkspace uses
    // the fixed RESERVED_WORKSPACE=16MB offset, so the sys region must be >= 16MB.
    uint32_t sysSize = static_cast<uint32_t>(ascendcPlatform->GetLibApiWorkSpaceSize());
    constexpr uint32_t QMM_RESERVED_WS_2201 = 16U * 1024U * 1024U;
    if (sysSize == 0U || sysSize == 0xFFFFFFFFU) { sysSize = QMM_RESERVED_WS_2201; }
    sysSize = ((sysSize + 511U) / 512U) * 512U;
    if (sysSize < QMM_RESERVED_WS_2201) { sysSize = QMM_RESERVED_WS_2201; }
    tilingData.sysWorkspaceSize = sysSize;

    uint64_t ringBytes = 0ULL;
    if (isPertoken != 0U) {
        ringBytes = static_cast<uint64_t>(QMM_RING_DEPTH) * usedCoreNum *
                    static_cast<uint64_t>(tilingData.cubeTilingData.baseM) *
                    static_cast<uint64_t>(tilingData.cubeTilingData.baseN) * sizeof(int32_t);
    }
    tilingData.workspaceSize = static_cast<uint32_t>(static_cast<uint64_t>(sysSize) + ringBytes);
    return tilingData;
}

// ===========================================================================
// QmmCubeKernel (template kernel.asc:523-652)
//   Process()    : no-pt write-through to y
//   ProcessRing(): pertoken per-base-tile ring producer (runtime tiling)
// ===========================================================================
template <typename AType, typename BType, typename CType, typename BiasType>
class QmmCubeKernel {
public:
    __aicore__ inline QmmCubeKernel() {}

    AscendC::Matmul<
        AscendC::MatmulType<AscendC::TPosition::GM, CubeFormat::ND, AType>,
        AscendC::MatmulType<AscendC::TPosition::GM, CubeFormat::NZ, BType>,
        AscendC::MatmulType<AscendC::TPosition::GM, CubeFormat::ND, CType>,
        AscendC::MatmulType<AscendC::TPosition::GM, CubeFormat::ND, BiasType>>
        matmulObj;

    __aicore__ inline void Init(__gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* c,
                                const TCubeTiling& tiling)
    {
        tiling_ = tiling;
        aGlobal_.SetGlobalBuffer(reinterpret_cast<__gm__ AType*>(a), (int64_t)tiling_.M * tiling_.Ka);
        bGlobal_.SetGlobalBuffer(reinterpret_cast<__gm__ BType*>(b), (int64_t)tiling_.Kb * tiling_.N);
        cGlobal_.SetGlobalBuffer(reinterpret_cast<__gm__ CType*>(c), (int64_t)tiling_.M * tiling_.N);
        CalcOffset(AscendC::GetBlockIdx(), offsetA_, offsetB_, offsetC_);
        aGlobal_ = aGlobal_[offsetA_];
        bGlobal_ = bGlobal_[offsetB_];
        cGlobal_ = cGlobal_[offsetC_];
    }

    __aicore__ inline void InitRing(__gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* ring,
                                    const TCubeTiling& tiling)
    {
        tiling_ = tiling;
        aGlobal_.SetGlobalBuffer(reinterpret_cast<__gm__ AType*>(a), (int64_t)tiling_.M * tiling_.Ka);
        bGlobal_.SetGlobalBuffer(reinterpret_cast<__gm__ BType*>(b), (int64_t)tiling_.Kb * tiling_.N);
        ringG_.SetGlobalBuffer(reinterpret_cast<__gm__ CType*>(ring),
                               (int64_t)QMM_RING_DEPTH * tiling_.usedCoreNum *
                               (int64_t)tiling_.baseM * tiling_.baseN);
        CalcOffset(AscendC::GetBlockIdx(), offsetA_, offsetB_, offsetC_);
        aGlobal_ = aGlobal_[offsetA_];
        bGlobal_ = bGlobal_[offsetB_];
    }

    __aicore__ inline void Process()
    {
        if (AscendC::GetBlockIdx() >= tiling_.usedCoreNum) return;
        int tailM = tiling_.M - mCoreIndex_ * tiling_.singleCoreM;
        tailM = tailM < tiling_.singleCoreM ? tailM : tiling_.singleCoreM;
        int tailN = tiling_.N - nCoreIndex_ * tiling_.singleCoreN;
        tailN = tailN < tiling_.singleCoreN ? tailN : tiling_.singleCoreN;
        if (tailM < tiling_.singleCoreM || tailN < tiling_.singleCoreN) {
            matmulObj.SetTail(tailM, tailN);
        }
        matmulObj.SetTensorA(aGlobal_, IS_TRANS_A);
        matmulObj.SetTensorB(bGlobal_, IS_TRANS_B);
        matmulObj.IterateAll(cGlobal_);
        matmulObj.End();
    }

    __aicore__ inline void ProcessRing()
    {
        if (AscendC::GetBlockIdx() >= tiling_.usedCoreNum) return;
        int tailM = tiling_.M - mCoreIndex_ * tiling_.singleCoreM;
        tailM = tailM < tiling_.singleCoreM ? tailM : tiling_.singleCoreM;
        int tailN = tiling_.N - nCoreIndex_ * tiling_.singleCoreN;
        tailN = tailN < tiling_.singleCoreN ? tailN : tiling_.singleCoreN;
        if (tailM < tiling_.singleCoreM || tailN < tiling_.singleCoreN) {
            matmulObj.SetTail(tailM, tailN);
        }
        matmulObj.SetTensorA(aGlobal_, IS_TRANS_A);
        matmulObj.SetTensorB(bGlobal_, IS_TRANS_B);
        const int32_t slotElems = tiling_.baseM * tiling_.baseN;
        int32_t tileIdx = 0;
        while (matmulObj.Iterate()) {
            if (tileIdx >= QMM_RING_DEPTH) {
                AscendC::CrossCoreWaitFlag<2>(QMM_AIV2AIC_RELEASE);
            }
            int64_t slotBase =
                (AscendC::GetBlockIdx() + (int64_t)(tileIdx % QMM_RING_DEPTH) * tiling_.usedCoreNum) * slotElems;
            matmulObj.GetTensorC(ringG_[slotBase], 0, true);
            AscendC::CrossCoreSetFlag<2, PIPE_FIX>(QMM_AIC2AIV_READY);
            tileIdx++;
        }
        int32_t drain = tileIdx < QMM_RING_DEPTH ? tileIdx : QMM_RING_DEPTH;
        for (int32_t i = 0; i < drain; ++i) {
            AscendC::CrossCoreWaitFlag<2>(QMM_AIV2AIC_RELEASE);
        }
        matmulObj.End();
    }

private:
    __aicore__ inline void CalcOffset(int32_t blockIdx, int64_t& offsetA, int64_t& offsetB, int64_t& offsetC)
    {
        const int32_t mSingleBlocks = (tiling_.M + tiling_.singleCoreM - 1) / tiling_.singleCoreM;
        mCoreIndex_ = blockIdx % mSingleBlocks;
        nCoreIndex_ = blockIdx / mSingleBlocks;
        offsetA = (int64_t)mCoreIndex_ * tiling_.singleCoreM * tiling_.Ka;
        offsetB = (int64_t)nCoreIndex_ * tiling_.singleCoreN * tiling_.Kb;
        offsetC = (int64_t)mCoreIndex_ * tiling_.N * tiling_.singleCoreM +
                  (int64_t)nCoreIndex_ * tiling_.singleCoreN;
    }

    TCubeTiling tiling_;
    GlobalTensor<AType> aGlobal_;
    GlobalTensor<BType> bGlobal_;
    GlobalTensor<CType> cGlobal_;
    GlobalTensor<CType> ringG_;
    int32_t mCoreIndex_ = 0;
    int32_t nCoreIndex_ = 0;
    int64_t offsetA_ = 0;
    int64_t offsetB_ = 0;
    int64_t offsetC_ = 0;
};

// ===========================================================================
// QmmPtCubeKernel (template kernel.asc:941-1028, pertoken AIC constant tiling)
// ===========================================================================
template <typename AType, typename BType, typename CType, typename BiasType>
class QmmPtCubeKernel {
public:
    __aicore__ inline QmmPtCubeKernel() {}
    using A_TYPE = AscendC::MatmulType<AscendC::TPosition::GM, CubeFormat::ND, AType>;
    using B_TYPE = AscendC::MatmulType<AscendC::TPosition::GM, CubeFormat::NZ, BType>;
    using C_TYPE = AscendC::MatmulType<AscendC::TPosition::GM, CubeFormat::ND, CType>;
    using BIAS_TYPE = AscendC::MatmulType<AscendC::TPosition::GM, CubeFormat::ND, BiasType>;

    constexpr static auto PT_CFG = qmm_pt_const::MakeCfg();
    constexpr static auto PT_CONST_TILING =
        AscendC::GetMatmulApiTiling<A_TYPE, B_TYPE, C_TYPE, BIAS_TYPE>(PT_CFG);
    constexpr static auto PT_MM_TILING = qmm_pt_const::MakeL1Tiling(PT_CONST_TILING);

    AscendC::Matmul<A_TYPE, B_TYPE, C_TYPE, BIAS_TYPE, PT_MM_TILING> matmulObj;

    __aicore__ inline void InitRing(__gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* ring,
                                    const TCubeTiling& tiling)
    {
        tiling_ = tiling;
        aGlobal_.SetGlobalBuffer(reinterpret_cast<__gm__ AType*>(a), (int64_t)tiling_.M * tiling_.Ka);
        bGlobal_.SetGlobalBuffer(reinterpret_cast<__gm__ BType*>(b), (int64_t)tiling_.Kb * tiling_.N);
        ringG_.SetGlobalBuffer(reinterpret_cast<__gm__ CType*>(ring),
                               (int64_t)QMM_RING_DEPTH * tiling_.usedCoreNum *
                               (int64_t)tiling_.baseM * tiling_.baseN);
        CalcOffset(AscendC::GetBlockIdx(), offsetA_, offsetB_, offsetC_);
        aGlobal_ = aGlobal_[offsetA_];
        bGlobal_ = bGlobal_[offsetB_];
    }

    __aicore__ inline void ProcessRing()
    {
        if (AscendC::GetBlockIdx() >= tiling_.usedCoreNum) return;
        int tailM = tiling_.M - mCoreIndex_ * tiling_.singleCoreM;
        tailM = tailM < tiling_.singleCoreM ? tailM : tiling_.singleCoreM;
        int tailN = tiling_.N - nCoreIndex_ * tiling_.singleCoreN;
        tailN = tailN < tiling_.singleCoreN ? tailN : tiling_.singleCoreN;
        matmulObj.SetOrgShape(tiling_.M, tiling_.N, tiling_.Ka);
        matmulObj.SetTail(tailM, tailN, tiling_.Ka);
        matmulObj.SetTensorA(aGlobal_, IS_TRANS_A);
        matmulObj.SetTensorB(bGlobal_, IS_TRANS_B);
        const int32_t slotElems = tiling_.baseM * tiling_.baseN;
        int32_t tileIdx = 0;
        while (matmulObj.Iterate()) {
            if (tileIdx >= QMM_RING_DEPTH) {
                AscendC::CrossCoreWaitFlag<2>(QMM_AIV2AIC_RELEASE);
            }
            int64_t slotBase =
                (AscendC::GetBlockIdx() + (int64_t)(tileIdx % QMM_RING_DEPTH) * tiling_.usedCoreNum) * slotElems;
            matmulObj.GetTensorC(ringG_[slotBase], 0, true);
            AscendC::CrossCoreSetFlag<2, PIPE_FIX>(QMM_AIC2AIV_READY);
            tileIdx++;
        }
        int32_t drain = tileIdx < QMM_RING_DEPTH ? tileIdx : QMM_RING_DEPTH;
        for (int32_t i = 0; i < drain; ++i) {
            AscendC::CrossCoreWaitFlag<2>(QMM_AIV2AIC_RELEASE);
        }
        matmulObj.End();
    }

private:
    __aicore__ inline void CalcOffset(int32_t blockIdx, int64_t& offsetA, int64_t& offsetB, int64_t& offsetC)
    {
        const int32_t mSingleBlocks = (tiling_.M + tiling_.singleCoreM - 1) / tiling_.singleCoreM;
        mCoreIndex_ = blockIdx % mSingleBlocks;
        nCoreIndex_ = blockIdx / mSingleBlocks;
        offsetA = (int64_t)mCoreIndex_ * tiling_.singleCoreM * tiling_.Ka;
        offsetB = (int64_t)nCoreIndex_ * tiling_.singleCoreN * tiling_.Kb;
        offsetC = (int64_t)mCoreIndex_ * tiling_.N * tiling_.singleCoreM +
                  (int64_t)nCoreIndex_ * tiling_.singleCoreN;
    }

    TCubeTiling tiling_;
    GlobalTensor<AType> aGlobal_;
    GlobalTensor<BType> bGlobal_;
    GlobalTensor<CType> ringG_;
    int32_t mCoreIndex_ = 0;
    int32_t nCoreIndex_ = 0;
    int64_t offsetA_ = 0;
    int64_t offsetB_ = 0;
    int64_t offsetC_ = 0;
};

// ===========================================================================
// QmmDequantVector (template kernel.asc:685-931, AIV row-batched dequant)
// ===========================================================================
class QmmDequantVector {
public:
    __aicore__ inline QmmDequantVector() {}
    __aicore__ inline void Init(__gm__ int32_t* ring, GM_ADDR scale, GM_ADDR pertoken, GM_ADDR y,
                                const TCubeTiling& tiling, AscendC::TPipe* pipe,
                                int32_t aivCombo, int32_t aivSub)
    {
        tiling_ = tiling;
        pipe_ = pipe;
        const int32_t mBlocks = (tiling_.M + tiling_.singleCoreM - 1) / tiling_.singleCoreM;
        mCoreIndex_ = aivCombo % mBlocks;
        nCoreIndex_ = aivCombo / mBlocks;
        m0_ = (int64_t)mCoreIndex_ * tiling_.singleCoreM;
        n0_ = (int64_t)nCoreIndex_ * tiling_.singleCoreN;
        rows_ = (int32_t)(tiling_.M - m0_);
        rows_ = rows_ < tiling_.singleCoreM ? rows_ : tiling_.singleCoreM;
        ncols_ = (int32_t)(tiling_.N - n0_);
        ncols_ = ncols_ < tiling_.singleCoreN ? ncols_ : tiling_.singleCoreN;
        slotElems_ = tiling_.baseM * tiling_.baseN;
        mIter_ = (rows_ + tiling_.baseM - 1) / tiling_.baseM;
        nIter_ = (ncols_ + tiling_.baseN - 1) / tiling_.baseN;
        tilesPerCore_ = mIter_ * nIter_;
        aivCombo_ = aivCombo;
        aivSub_ = aivSub;

        ringG_.SetGlobalBuffer(ring, (int64_t)QMM_RING_DEPTH * tiling_.usedCoreNum * slotElems_);
        scaleGm_.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(scale), tiling_.N);
        pertokenGm_.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(pertoken), tiling_.M);
        yGm_.SetGlobalBuffer(reinterpret_cast<__gm__ bfloat16_t*>(y), (int64_t)tiling_.M * tiling_.N);

        int32_t rowCap = tiling_.baseN;
        if (rowCap > (int32_t)QMM_DEQ_CHUNK_N) { rowCap = (int32_t)QMM_DEQ_CHUNK_N; }
        rowCap_ = rowCap;
        bandMax_ = (tiling_.baseM + 1) / 2;
        if (bandMax_ < 1) { bandMax_ = 1; }
        const int32_t pRound = ((bandMax_ + 7) / 8) * 8;
        const int32_t pBrElems = ((bandMax_ + 7) / 8) * 64;
        pRound_ = pRound;
        pBrElems_ = pBrElems;
        const int32_t ubBudget = 176 * 1024;
        const int32_t fixedBytes = rowCap_ * (int32_t)sizeof(float) +
                                   pRound_ * (int32_t)sizeof(float) +
                                   pBrElems_ * (int32_t)sizeof(float);
        int32_t eCap = (ubBudget - fixedBytes) / 10;
        if (eCap < rowCap_) { eCap = rowCap_; }
        const int32_t maxElems = bandMax_ * rowCap_;
        if (eCap > maxElems) { eCap = maxElems; }
        elemsCap_ = eCap;

        pipe_->InitBuffer(cInQ_, 1, elemsCap_ * sizeof(int32_t));
        pipe_->InitBuffer(scaleInQ_, 1, rowCap_ * sizeof(float));
        pipe_->InitBuffer(cFltBuf_, elemsCap_ * sizeof(float));
        pipe_->InitBuffer(yOutQ_, 1, elemsCap_ * sizeof(bfloat16_t));
        pipe_->InitBuffer(pTmpQ_, 1, pRound_ * sizeof(float));
        pipe_->InitBuffer(pBrcbBuf_, pBrElems_ * sizeof(float));
    }

    __aicore__ inline void Process()
    {
        LocalTensor<float> cFlt = cFltBuf_.Get<float>();
        const int32_t itOrder = tiling_.iterateOrder;
        for (int32_t t = 0; t < tilesPerCore_; ++t) {
            AscendC::CrossCoreWaitFlag<2>(QMM_AIC2AIV_READY);
            const int64_t slotBase =
                (aivCombo_ + (int64_t)(t % QMM_RING_DEPTH) * tiling_.usedCoreNum) * slotElems_;
            int32_t rt = 0, ct = 0;
            if (itOrder == 0) { rt = t % mIter_; ct = t / mIter_; }
            else { rt = t / nIter_; ct = t % nIter_; }
            const int32_t gRow0 = (int32_t)(m0_ + (int64_t)rt * tiling_.baseM);
            const int32_t gCol0 = (int32_t)(n0_ + (int64_t)ct * tiling_.baseN);
            const int32_t rr = (rows_ - rt * tiling_.baseM) < tiling_.baseM
                                   ? (rows_ - rt * tiling_.baseM) : tiling_.baseM;
            const int32_t rc = (ncols_ - ct * tiling_.baseN) < tiling_.baseN
                                   ? (ncols_ - ct * tiling_.baseN) : tiling_.baseN;
            if (rr <= 0 || rc <= 0) {
                AscendC::CrossCoreSetFlag<2, PIPE_MTE2>(QMM_AIV2AIC_RELEASE);
                continue;
            }
            int32_t r0 = 0, r1 = rr, c0 = 0, c1 = rc;
            if (rr >= 2) {
                const int32_t half = (rr + 1) / 2;
                if (aivSub_ == 0) { r1 = half; } else { r0 = half; }
            } else {
                const int32_t half = (rc + 1) / 2;
                if (aivSub_ == 0) { c1 = half; } else { c0 = half; }
            }
            if (r0 >= r1 || c0 >= c1) {
                AscendC::CrossCoreSetFlag<2, PIPE_MTE2>(QMM_AIV2AIC_RELEASE);
                continue;
            }
            const int32_t band = r1 - r0;
            const int32_t width = c1 - c0;
            const int64_t srcOff = slotBase + (int64_t)r0 * rc + c0;

            LocalTensor<float> scaleIn = scaleInQ_.AllocTensor<float>();
            DataCopy(scaleIn, scaleGm_[gCol0 + c0], width);
            scaleInQ_.EnQue(scaleIn);
            scaleIn = scaleInQ_.DeQue<float>();

            const uint8_t fullRep = (uint8_t)(width / 64);
            const uint32_t tail = (uint32_t)(width % 64);
            BinaryRepeatParams scaleP(1, 1, 1, 8, 8, 8);
            BinaryRepeatParams ptP(1, 1, 0, 8, 8, 0);

            int32_t rowsPerChunk = elemsCap_ / width;
            if (rowsPerChunk < 1) { rowsPerChunk = 1; }
            for (int32_t rStart = 0; rStart < band; rStart += rowsPerChunk) {
                int32_t rcRows = band - rStart;
                if (rcRows > rowsPerChunk) { rcRows = rowsPerChunk; }
                const int32_t elems = rcRows * width;
                LocalTensor<int32_t> cIn = cInQ_.AllocTensor<int32_t>();
                DataCopy(cIn, ringG_[srcOff + (int64_t)rStart * rc], elems);
                cInQ_.EnQue(cIn);
                cIn = cInQ_.DeQue<int32_t>();
                const int32_t pRoundRows = ((rcRows + 7) / 8) * 8;
                LocalTensor<float> pRaw = pTmpQ_.AllocTensor<float>();
                DataCopyExtParams pcp;
                pcp.blockCount = 1;
                pcp.blockLen = (uint32_t)rcRows * sizeof(float);
                pcp.srcStride = 0;
                pcp.dstStride = 0;
                pcp.rsv = 0;
                DataCopyPadExtParams<float> pp(false, 0, 0, (float)0);
                DataCopyPad(pRaw, pertokenGm_[(int64_t)(gRow0 + r0 + rStart)], pcp, pp);
                pTmpQ_.EnQue(pRaw);
                pRaw = pTmpQ_.DeQue<float>();
                LocalTensor<float> pBr = pBrcbBuf_.Get<float>();
                Brcb(pBr, pRaw, (uint8_t)(pRoundRows / 8), BrcbRepeatParams(1, 8));
                pTmpQ_.FreeTensor(pRaw);

                Cast(cFlt, cIn, AscendC::RoundMode::CAST_RINT, elems);
                for (int32_t i = 0; i < rcRows; ++i) {
                    const int64_t off = (int64_t)i * width;
                    if (fullRep > 0) {
                        Mul(cFlt[off], cFlt[off], scaleIn, (uint64_t)64, fullRep, scaleP);
                        Mul(cFlt[off], cFlt[off], pBr[i * 8], (uint64_t)64, fullRep, ptP);
                    }
                    if (tail > 0) {
                        Mul(cFlt[off + fullRep * 64], cFlt[off + fullRep * 64],
                            scaleIn[fullRep * 64], (uint64_t)tail, (uint8_t)1, scaleP);
                        Mul(cFlt[off + fullRep * 64], cFlt[off + fullRep * 64],
                            pBr[i * 8], (uint64_t)tail, (uint8_t)1, ptP);
                    }
                }
                LocalTensor<bfloat16_t> yOut = yOutQ_.AllocTensor<bfloat16_t>();
                Cast(yOut, cFlt, AscendC::RoundMode::CAST_RINT, elems);
                yOutQ_.EnQue(yOut);
                yOut = yOutQ_.DeQue<bfloat16_t>();
                DataCopyParams yp;
                yp.blockCount = (uint16_t)rcRows;
                yp.blockLen = (uint16_t)(width * sizeof(bfloat16_t) / 32);
                yp.srcStride = 0;
                yp.dstStride = (uint16_t)((tiling_.N - width) * sizeof(bfloat16_t) / 32);
                const int64_t yOff = (int64_t)(gRow0 + r0 + rStart) * tiling_.N + gCol0 + c0;
                DataCopy(yGm_[yOff], yOut, yp);
                yOutQ_.FreeTensor(yOut);
                cInQ_.FreeTensor(cIn);
            }
            scaleInQ_.FreeTensor(scaleIn);
            AscendC::CrossCoreSetFlag<2, PIPE_MTE2>(QMM_AIV2AIC_RELEASE);
        }
    }

private:
    TCubeTiling tiling_;
    int32_t mCoreIndex_ = 0;
    int32_t nCoreIndex_ = 0;
    int64_t m0_ = 0;
    int64_t n0_ = 0;
    int32_t rows_ = 0;
    int32_t ncols_ = 0;
    int32_t slotElems_ = 0;
    int32_t mIter_ = 0;
    int32_t nIter_ = 0;
    int32_t tilesPerCore_ = 0;
    int32_t aivCombo_ = 0;
    int32_t aivSub_ = 0;
    int32_t rowCap_ = 0;
    int32_t bandMax_ = 0;
    int32_t elemsCap_ = 0;
    int32_t pRound_ = 0;
    int32_t pBrElems_ = 0;
    AscendC::TPipe* pipe_ = nullptr;
    GlobalTensor<int32_t> ringG_;
    GlobalTensor<float> scaleGm_;
    GlobalTensor<float> pertokenGm_;
    GlobalTensor<bfloat16_t> yGm_;
    TQue<TPosition::VECIN, 1> cInQ_;
    TQue<TPosition::VECIN, 1> scaleInQ_;
    TBuf<TPosition::VECCALC> cFltBuf_;
    TQue<TPosition::VECOUT, 1> yOutQ_;
    TQue<TPosition::VECIN, 1> pTmpQ_;
    TBuf<TPosition::VECCALC> pBrcbBuf_;
};

// Shared AIV entry (template kernel.asc:1031-1045)
__aicore__ inline void QmmRunDequantAiv(GM_ADDR ringWs, GM_ADDR scale, GM_ADDR pertoken_scale,
                                        GM_ADDR y, const TCubeTiling& tiling, AscendC::TPipe* pipe)
{
    const int64_t subNum = AscendC::GetSubBlockNum();
    const int64_t aivLogic = AscendC::GetBlockIdx();
    const int64_t combo = aivLogic / subNum;
    const int64_t sub = AscendC::GetSubBlockIdx();
    if (combo >= tiling.usedCoreNum) return;
    QmmDequantVector op;
    op.Init(reinterpret_cast<__gm__ int32_t*>(ringWs), scale, pertoken_scale, y,
            tiling, pipe, (int32_t)combo, (int32_t)sub);
    op.Process();
}

// ===========================================================================
// Task 2: A8W8 -> INT32 output (scaffold signature)
// ===========================================================================
class QmmCubeBasicKernel {
public:
    __aicore__ inline QmmCubeBasicKernel() {}

    __aicore__ inline void Init(GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR y,
                                GM_ADDR workSpace, const QmmCustomTilingData *__restrict tilingData, AscendC::TPipe *tPipe)
    {
        x1_ = x1; x2_ = x2; y_ = y; ws_ = workSpace;
        tilingData_ = tilingData; pipe_ = tPipe;
        if ASCEND_IS_AIC {
            if (qmm_cfg::kSetSysWorkspace) {
                // Direct launch without HAVE_WORKSPACE: kernel must set sys workspace.
                AscendC::SetSysWorkspace(workSpace);
                // On 2201 SetSysWorkspace only writes g_sysWorkspaceReserved; force-bind
                // KFC when it is still null.
                if (qmm_cfg::kForceSysWorkspaceFallback && GetSysWorkSpacePtr() == nullptr) {
                    AscendC::SetSysWorkspaceForce(workSpace);
                }
            }
        }
    }

    __aicore__ inline void Process()
    {
        if ASCEND_IS_AIV { return; }                       // Cube-only path
        if (AscendC::GetBlockIdx() >= tilingData_->cubeTilingData.usedCoreNum) { return; }
        AscendC::InitSocState();
        const int64_t M = tilingData_->M, N = tilingData_->N, K = tilingData_->K;
        if (qmm_cfg::kUseL2C && QmmL2CShapeHit(M, K, N)) {
            // O4: L2CacheOptimizer + constant tiling, write-through to y
            qmm_l2c::QmmL2cKernel<int8_t, int8_t, int32_t, int32_t> cube;
            REGIST_MATMUL_OBJ(pipe_, GetSysWorkSpacePtr(), cube.matmulObj, (TCubeTiling*)nullptr);
            cube.Init(x1_, x2_, y_, (int32_t)M, (int32_t)N, (int32_t)K);
            cube.Process();
            return;
        }
        QmmCubeKernel<int8_t, int8_t, int32_t, int32_t> cube;
        REGIST_MATMUL_OBJ(pipe_, GetSysWorkSpacePtr(), cube.matmulObj, &tilingData_->cubeTilingData);
        cube.Init(x1_, x2_, y_, tilingData_->cubeTilingData);
        cube.Process();
    }

private:
    GM_ADDR x1_ = nullptr;
    GM_ADDR x2_ = nullptr;
    GM_ADDR y_ = nullptr;
    GM_ADDR ws_ = nullptr;
    const QmmCustomTilingData *tilingData_ = nullptr;
    AscendC::TPipe *pipe_ = nullptr;
};

// ===========================================================================
// Task 3: A8W8 + perChannel/perToken dequant -> BF16 (scaffold signature)
// ===========================================================================
class QmmPertokenKernel {
public:
    __aicore__ inline QmmPertokenKernel() {}

    __aicore__ inline void Init(GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR pertokenScale, GM_ADDR y,
                                GM_ADDR workSpace, const QmmCustomTilingData *__restrict tilingData,
                                AscendC::TPipe *tPipe)
    {
        x1_ = x1; x2_ = x2; scale_ = scale; pertokenScale_ = pertokenScale; y_ = y;
        ws_ = workSpace; tilingData_ = tilingData; pipe_ = tPipe;
        if ASCEND_IS_AIC {
            if (qmm_cfg::kSetSysWorkspace) {
                AscendC::SetSysWorkspace(workSpace);
                if (qmm_cfg::kForceSysWorkspaceFallback && GetSysWorkSpacePtr() == nullptr) {
                    AscendC::SetSysWorkspaceForce(workSpace);
                }
            }
        }
    }

    __aicore__ inline void Process()
    {
        AscendC::InitSocState();
        GM_ADDR ringWs = ws_ + tilingData_->sysWorkspaceSize;   // user ring after sys prefix
        if ASCEND_IS_AIC {
            if (AscendC::GetBlockIdx() >= tilingData_->cubeTilingData.usedCoreNum) { return; }
            const TCubeTiling& tiling = tilingData_->cubeTilingData;
            const bool ptConst = qmm_cfg::kUsePtConst &&
                                 (tilingData_->M >= 512) && (tilingData_->M <= 30000) &&
                                 (tilingData_->N <= 30000) && (tilingData_->K <= 30000) &&
                                 (tiling.baseM == 128) && (tiling.baseN == 256) && (tiling.iterateOrder == 0);
            if (ptConst) {
                QmmPtCubeKernel<int8_t, int8_t, int32_t, int32_t> cube;
                REGIST_MATMUL_OBJ(pipe_, GetSysWorkSpacePtr(), cube.matmulObj, (TCubeTiling*)nullptr);
                cube.InitRing(x1_, x2_, ringWs, tiling);
                cube.ProcessRing();
            } else {
                QmmCubeKernel<int8_t, int8_t, int32_t, int32_t> cube;
                REGIST_MATMUL_OBJ(pipe_, GetSysWorkSpacePtr(), cube.matmulObj, &tilingData_->cubeTilingData);
                cube.InitRing(x1_, x2_, ringWs, tiling);
                cube.ProcessRing();
            }
        } else {
            // AIV: consume ring slots and dequantize
            QmmRunDequantAiv(ringWs, scale_, pertokenScale_, y_, tilingData_->cubeTilingData, pipe_);
        }
    }

private:
    GM_ADDR x1_ = nullptr;
    GM_ADDR x2_ = nullptr;
    GM_ADDR scale_ = nullptr;
    GM_ADDR pertokenScale_ = nullptr;
    GM_ADDR y_ = nullptr;
    GM_ADDR ws_ = nullptr;
    const QmmCustomTilingData *tilingData_ = nullptr;
    AscendC::TPipe *pipe_ = nullptr;
};


__global__ __mix__(1, 2) __aicore__
    void qmm_custom_kernel(GM_ADDR x1, GM_ADDR x2, GM_ADDR scale,
                           GM_ADDR pertoken_scale, GM_ADDR y, GM_ADDR workspace,
                           QmmCustomTilingData tilingData) {
  if (workspace == nullptr) {
    return;
  }
  AscendC::TPipe pipe;

  if (tilingData.isPertoken == 0) {
    QmmCubeBasicKernel kernel;
    kernel.Init(x1, x2, scale, y, workspace, &tilingData, &pipe);
    kernel.Process();
    pipe.Destroy();
  } else {
    QmmPertokenKernel kernel;
    kernel.Init(x1, x2, scale, pertoken_scale, y, workspace, &tilingData,
                &pipe);
    kernel.Process();
    pipe.Destroy();
  }
}

#ifndef ACL_PROF_TENSOR_INFO_DEFINED
#define ACL_PROF_TENSOR_INFO_DEFINED
struct aclprofTensor {
  uint32_t type;     // Tensor类型: 0表示输入, 1表示输出。
  uint32_t format;   // Format类型, 对应aclFormat。
  uint32_t dataType; // 数据类型, 对应aclDataType。
  uint32_t shapeDim; // Shape维度数量, 最大支持8维。
  uint32_t shape[8]; // Shape信息。
};

struct aclprofTensorInfo {
  uint64_t opNameId; // 通过uint64_t aclprofStr2Id(const char *message)转化
  uint64_t opTypeId; // 通过uint64_t aclprofStr2Id(const char *message)转化
  uint32_t resv;
  uint32_t tensorNum;
  uint32_t kernelType;
  uint32_t blockNums;
  void *stream;
  aclprofTensor *tensors;
};

struct aclprofEventAttributes {
  uint16_t version;
  uint16_t size;
  uint32_t messageType;
  union message {
    aclprofTensorInfo *tensorInfo;
  } message;
};
#endif // ACL_PROF_TENSOR_INFO_DEFINED

#define INPUT(x)                                                               \
  aclprofTensor {                                                              \
    (uint32_t)0,                                                               \
        static_cast<uint32_t>(                                                 \
            ascendc_ops::GetFormat(static_cast<int64_t>(x.sizes().size()))),   \
        static_cast<uint32_t>(ascendc_ops::GetAclDataType(x.scalar_type())),   \
        static_cast<uint32_t>(x.sizes().size()), {                             \
      static_cast<uint32_t>(x.sizes().size() > 0 ? x.sizes().data()[0] : 0),   \
          static_cast<uint32_t>(x.sizes().size() > 1 ? x.sizes().data()[1]     \
                                                     : 0),                     \
          static_cast<uint32_t>(x.sizes().size() > 2 ? x.sizes().data()[2]     \
                                                     : 0),                     \
          static_cast<uint32_t>(x.sizes().size() > 3 ? x.sizes().data()[3]     \
                                                     : 0)                      \
    }                                                                          \
  }

#define OUTPUT(z)                                                              \
  aclprofTensor {                                                              \
    (uint32_t)1,                                                               \
        static_cast<uint32_t>(                                                 \
            ascendc_ops::GetFormat(static_cast<int64_t>(z.sizes().size()))),   \
        static_cast<uint32_t>(ascendc_ops::GetAclDataType(z.scalar_type())),   \
        static_cast<uint32_t>(z.sizes().size()), {                             \
      static_cast<uint32_t>(z.sizes().size() > 0 ? z.sizes().data()[0] : 0),   \
          static_cast<uint32_t>(z.sizes().size() > 1 ? z.sizes().data()[1]     \
                                                     : 0),                     \
          static_cast<uint32_t>(z.sizes().size() > 2 ? z.sizes().data()[2]     \
                                                     : 0),                     \
          static_cast<uint32_t>(z.sizes().size() > 3 ? z.sizes().data()[3]     \
                                                     : 0)                      \
    }                                                                          \
  }

#define INIT_ACL_PROF_TENSOR_INFO(opName, opType, blockdim, kernelType,        \
                                  tensorInfo, stream, ...)                     \
  uint64_t opNameId = aclprofStr2Id(opName);                                   \
  uint64_t opTypeId = aclprofStr2Id(opType);                                   \
  struct aclprofTensor tensors[] = {__VA_ARGS__};                              \
  tensorInfo = {                                                               \
      opNameId,   opTypeId, 0,      sizeof(tensors) / sizeof(aclprofTensor),   \
      kernelType, blockdim, stream, tensors}

namespace ascendc_ops {

aclDataType GetAclDataType(at::ScalarType scalarType) {
  switch (scalarType) {
  case at::ScalarType::Float:
    return ACL_FLOAT;
  case at::ScalarType::Int:
    return ACL_INT32;
  case at::ScalarType::BFloat16:
    return ACL_BF16;
  case at::ScalarType::Byte:
    return ACL_UINT8;
  case at::ScalarType::Char:
    return ACL_INT8;
  default:
    TORCH_CHECK(false,
                "Unsupported tensor dtype for profiling report: ", scalarType);
  }
}

aclFormat GetFormat(int64_t dimNum) {
  aclFormat format = ACL_FORMAT_ND;
  if (dimNum == 4) {
    format = ACL_FORMAT_FRACTAL_NZ;
  }
  return format;
}

at::Tensor qmm_custom(const at::Tensor &x1, const at::Tensor &x2,
                      const at::Tensor &scale,
                      const c10::optional<at::Tensor> &pertoken_scale) {
  auto aclStream = c10_npu::getCurrentNPUStream().stream(true);

  auto M = x1.size(0);
  auto K = x1.size(1);
  auto N = x2.size(1);

  uint32_t isPertoken =
      pertoken_scale.has_value() && pertoken_scale.value().defined() ? 1 : 0;
  at::ScalarType output_dtype =
      isPertoken ? at::ScalarType::BFloat16 : at::ScalarType::Int;
  at::Tensor y = at::empty({M, N}, x1.options().dtype(output_dtype));

  QmmCustomTilingData tilingData = CalcQmmTiling(isPertoken, M, N, K);
  uint32_t numBlocks = tilingData.cubeTilingData.usedCoreNum;

  at::Tensor workspaceTensor =
      at::empty({static_cast<int64_t>(tilingData.workspaceSize)},
                x1.options().dtype(at::ScalarType::Byte));

  aclprofTensorInfo tensorInfo;
  uint64_t opNameId = aclprofStr2Id("qmm_custom");
  uint64_t opTypeId = aclprofStr2Id("QmmCustom");
  uint8_t *pertokenPtr =
      isPertoken ? (uint8_t *)(pertoken_scale.value().mutable_data_ptr())
                 : nullptr;
  std::vector<aclprofTensor> tensors = {INPUT(x1), INPUT(x2), INPUT(scale)};
  if (isPertoken) {
    tensors.push_back(INPUT(pertoken_scale.value()));
  }
  tensors.push_back(OUTPUT(y));
  tensorInfo = {opNameId, opTypeId,  0,         (uint32_t)tensors.size(),
                0,        numBlocks, aclStream, tensors.data()};

  aclprofEventAttributes attrs = {1, sizeof(aclprofEventAttributes::message), 0,
                                  &tensorInfo};

  aclprofRangePushEx(&attrs);
  qmm_custom_kernel<<<numBlocks, 0, aclStream>>>(
      (uint8_t *)(x1.mutable_data_ptr()), (uint8_t *)(x2.mutable_data_ptr()),
      (uint8_t *)(scale.mutable_data_ptr()), pertokenPtr,
      (uint8_t *)(y.mutable_data_ptr()), workspaceTensor.data_ptr<uint8_t>(),
      tilingData);
  aclprofRangePop();
  return y;
}
} // namespace ascendc_ops

TORCH_LIBRARY(ascendc_ops, m) {
  m.def("qmm_custom(Tensor x1, Tensor x2, Tensor scale, Tensor? "
        "pertoken_scale=None) -> Tensor");
}

TORCH_LIBRARY_IMPL(ascendc_ops, PrivateUse1, m) {
  m.impl("qmm_custom", TORCH_FN(ascendc_ops::qmm_custom));
}


Overwriting /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/qmm_custom.asc


---
## 4 编译自定义算子

### 任务四：成功编译自定义算子


在算子目录下执行编译命令：

In [3]:
!cd {QMM_CUSTOM_DIR} && mkdir -p build && cd build && cmake -DCMAKE_ASC_ARCHITECTURES=dav-2201 .. && make -j;

CMake Warning at /mnt/workspace/.venv/lib/python3.11/site-packages/torch/share/cmake/Torch/TorchConfig.cmake:22 (message):
  static library kineto_LIBRARY-NOTFOUND not found.
Call Stack (most recent call first):
  /mnt/workspace/.venv/lib/python3.11/site-packages/torch/share/cmake/Torch/TorchConfig.cmake:125 (append_torchlib_if_found)
  CMakeLists.txt:14 (find_package)




-- Configuring done
-- Generating done
-- Build files have been written to: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/build
Consolidate compiler generated dependencies of target ascendc_ops


[ 50%] Building ASC object CMakeFiles/ascendc_ops.dir/qmm_custom.asc.o


/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/qmm_custom.asc:808:26: warning: 'SetSysWorkspace' is deprecated: NOTICE: SetSysWorkSpace has been deprecated and will be removed in the next version. [-Wdeprecated-declarations]
                AscendC::SetSysWorkspace(workSpace);
                         ^
/home/developer/Ascend/cann-9.0.0/aarch64-linux/asc/impl/basic_api/dav_c220/kernel_operator_common_impl.h:28:3: note: 'SetSysWorkspace' has been explicitly marked deprecated here
[[deprecated(
  ^


/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/qmm_custom.asc:862:26: warning: 'SetSysWorkspace' is deprecated: NOTICE: SetSysWorkSpace has been deprecated and will be removed in the next version. [-Wdeprecated-declarations]
                AscendC::SetSysWorkspace(workSpace);
                         ^
/home/developer/Ascend/cann-9.0.0/aarch64-linux/asc/impl/basic_api/dav_c220/kernel_operator_common_impl.h:28:3: note: 'SetSysWorkspace' has been explicitly marked deprecated here
[[deprecated(
  ^


...

/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/qmm_custom.asc:808:26: warning: 'SetSysWorkspace' is deprecated: NOTICE: SetSysWorkSpace has been deprecated and will be removed in the next version. [-Wdeprecated-declarations]
                AscendC::SetSysWorkspace(workSpace);
                         ^
/home/developer/Ascend/cann-9.0.0/aarch64-linux/asc/impl/basic_api/dav_c220/kernel_operator_common_impl.h:28:3: note: 'SetSysWorkspace' has been explicitly marked deprecated here
[[deprecated(
  ^


/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/qmm_custom.asc:862:26: warning: 'SetSysWorkspace' is deprecated: NOTICE: SetSysWorkSpace has been deprecated and will be removed in the next version. [-Wdeprecated-declarations]
                AscendC::SetSysWorkspace(workSpace);
                         ^
/home/developer/Ascend/cann-9.0.0/aarch64-linux/asc/impl/basic_api/dav_c220/kernel_operator_common_impl.h:28:3: note: 'SetSysWorkspace' has been explicitly marked deprecated here
[[deprecated(
  ^


2 warnings generated.


[100%] Linking ASC shared library libascendc_ops.so


[100%] Built target ascendc_ops


编译成功后会在build目录下生成ascendc_ops.so

---
## 5 测试自定义算子功能性能

### 任务五：根据提供的测试脚本，测试自定义算子功能与性能，尝试对自定义算子进行优化后重复整个流程

使用测试脚本验证 QmmCustom 算子在不同规格输入下的精度和性能。测试脚本覆盖多组 (M, K, N) 规格，分别验证 INT32 输出（无 perTokenScale）和 BF16 输出（有 perTokenScale）两种路径。

核心接口调用如下：

```python
output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu, pertoken_scale_npu)
```

### 5.1 测试算子功能

需要测试的 cases 的 shape 如下，对应的是 Qwen3-8B 模型中实际出现的调用，**全部通过**视为算子功能OK：
| M    | K     | N     |
| ---- | ----- | ----- |
| 1    | 4096  | 4096  |
| 1    | 4096  | 6144  |
| 1    | 4096  | 24576 |
| 1    | 12288 | 4096  |
| 50   | 4096  | 4096  |
| 50   | 4096  | 6144  |
| 50   | 4096  | 24576 |
| 50   | 12288 | 4096  |
| 4096 | 4096  | 4096  |
| 4096 | 4096  | 6144  |
| 4096 | 4096  | 24576 |
| 4096 | 12288 | 4096  |

In [4]:
import torch
import torch_npu
torch.ops.load_library(str(QMM_CUSTOM_DIR / "build/libascendc_ops.so"))

torch.npu.config.allow_internal_format = True
torch.npu.config.allow_hf32 = False
torch.npu.set_compile_mode(jit_compile=False)

# ============================================================
# 测试配置: 多组 (M, K, N) 规格
# ============================================================
TEST_SHAPES = [
    (1, 4096, 4096),
    (1, 4096, 6144),
    (1, 4096, 24576),
    (1, 12288, 4096),
    (50, 4096, 4096),
    (50, 4096, 6144),
    (50, 4096, 24576),
    (50, 12288, 4096),
    (4096, 4096, 4096),
    (4096, 4096, 6144),
    (4096, 4096, 24576),
    (4096, 12288, 4096),
]


def check_close(output, ref, dtype):
    """统一的精度比对函数，根据输出 dtype 使用不同的 tolerance"""
    if dtype == torch.int32:
        # 整数运算应精确匹配，tolerance 为 0
        rtol, atol = 0, 0
    else:
        # BF16 允许浮点舍入误差
        rtol, atol = 0.01, 0.01

    output_f = output.to(torch.float32)
    ref_f = ref.to(torch.float32)
    passed = torch.allclose(output_f, ref_f, rtol=rtol, atol=atol)

    label = 'INT32' if dtype == torch.int32 else 'BF16'
    print(f"  {label} allclose验证: {'PASS' if passed else 'FAIL'}")

    if not passed:
        print(f"  output[0,0:5] = {output[0, 0:5].tolist()}")
        print(f"  ref[0,0:5]    = {ref[0, 0:5].tolist()}")

    return passed


def test_int32(M, K, N):
    """无 pertoken_scale, 纯INT8 matmul, INT32 输出"""
    x1_cpu = torch.randint(-128, 127, (M, K), dtype=torch.int8)
    x2_cpu = torch.randint(-128, 127, (K, N), dtype=torch.int8)
    scale_cpu = torch.randn([N], dtype=torch.float32)

    x1_npu = x1_cpu.npu()
    x2_npu = x2_cpu.npu()
    x2_npu = torch_npu.npu_format_cast(x2_npu, 29)  # 29: FORMAT_FRACTAL_NZ
    scale_npu = scale_cpu.npu()

    output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu, pertoken_scale=None)
    output_cpu = output_npu.cpu()

    # float64 matmul 走 BLAS 优化, 比 int32 朴素乘加快很多;
    # float64 精度足够精确表示 int32 范围内的整数结果
    ref_output = torch.matmul(x1_cpu.to(torch.float64), x2_cpu.to(torch.float64)).round().to(torch.int32)

    assert output_cpu.shape == ref_output.shape, f"shape不一致: {output_cpu.shape} vs {ref_output.shape}"
    assert output_cpu.dtype == torch.int32, f"输出dtype不正确: {output_cpu.dtype}"

    return check_close(output_cpu, ref_output, torch.int32)


def test_bf16(M, K, N):
    """有 pertoken_scale, INT8 matmul + dequant, BF16 输出"""
    x1_cpu = torch.randint(-128, 127, (M, K), dtype=torch.int8)
    x2_cpu = torch.randint(-128, 127, (K, N), dtype=torch.int8)
    scale_cpu = torch.randn([N], dtype=torch.float32)
    pertoken_scale_cpu = torch.randn([M], dtype=torch.float32)

    x1_npu = x1_cpu.npu()
    x2_npu = x2_cpu.npu()
    x2_npu = torch_npu.npu_format_cast(x2_npu, 29)  # 29: FORMAT_FRACTAL_NZ
    scale_npu = scale_cpu.npu()
    pertoken_scale_npu = pertoken_scale_cpu.npu()

    output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu,
                                                  pertoken_scale=pertoken_scale_npu)
    output_cpu = output_npu.cpu()

    int32_result = torch.matmul(x1_cpu.to(torch.float64), x2_cpu.to(torch.float64)).round().to(torch.int32)
    ref_bf16 = (int32_result.to(torch.float32) * scale_cpu.unsqueeze(0) * pertoken_scale_cpu.unsqueeze(1)).to(torch.bfloat16)

    assert output_cpu.shape == ref_bf16.shape, f"shape不一致: {output_cpu.shape} vs {ref_bf16.shape}"
    assert output_cpu.dtype == torch.bfloat16, f"输出dtype不正确: {output_cpu.dtype}"

    return check_close(output_cpu, ref_bf16, torch.bfloat16)


# ============================================================
# 主循环: 对每组 (M, K, N) 执行 INT32 和 BF16 测试
# ============================================================
int32_results = []
bf16_results = []

for i, (M, K, N) in enumerate(TEST_SHAPES, 1):
    print("=" * 60)
    print(f"规格{i}: M={M}, K={K}, N={N}")
    print("=" * 60)

    print("[INT32 测试]")
    int32_pass = test_int32(M, K, N)
    int32_results.append((M, K, N, int32_pass))

    print("[BF16 测试]")
    bf16_pass = test_bf16(M, K, N)
    bf16_results.append((M, K, N, bf16_pass))

# ============================================================
# 测试汇总
# ============================================================
print()
print("=" * 60)
print("测试汇总")
print("=" * 60)
print(f"{'规格':>24} | {'INT32':^8} | {'BF16':^8}")
print("-" * 46)
for (M, K, N, p1), (_, _, _, p2) in zip(int32_results, bf16_results):
    tag = f"M={M},K={K},N={N}"
    print(f"{tag:>24} | {'PASS' if p1 else 'FAIL':^8} | {'PASS' if p2 else 'FAIL':^8}")

int32_total = sum(p for _, _, _, p in int32_results)
bf16_total = sum(p for _, _, _, p in bf16_results)
total = len(TEST_SHAPES)
print(f"INT32: {int32_total}/{total} 通过, BF16: {bf16_total}/{total} 通过")
print("=" * 60)


/mnt/workspace/.venv/lib/python3.11/site-packages/torch_npu/__init__.py:320: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default.                      Do not set it to 1 to prevent some unknown errors
  warnings.warn("On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. \


规格1: M=1, K=4096, N=4096
[INT32 测试]


  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格2: M=1, K=4096, N=6144
[INT32 测试]


  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格3: M=1, K=4096, N=24576
[INT32 测试]


  INT32 allclose验证: PASS
[BF16 测试]


...

  BF16 allclose验证: PASS
规格11: M=4096, K=4096, N=24576
[INT32 测试]


  INT32 allclose验证: PASS
[BF16 测试]


  BF16 allclose验证: PASS
规格12: M=4096, K=12288, N=4096
[INT32 测试]


  INT32 allclose验证: PASS
[BF16 测试]


  BF16 allclose验证: PASS

测试汇总
                      规格 |  INT32   |   BF16  
----------------------------------------------
       M=1,K=4096,N=4096 |   PASS   |   PASS  
       M=1,K=4096,N=6144 |   PASS   |   PASS  
      M=1,K=4096,N=24576 |   PASS   |   PASS  
      M=1,K=12288,N=4096 |   PASS   |   PASS  
      M=50,K=4096,N=4096 |   PASS   |   PASS  
      M=50,K=4096,N=6144 |   PASS   |   PASS  
     M=50,K=4096,N=24576 |   PASS   |   PASS  
     M=50,K=12288,N=4096 |   PASS   |   PASS  
    M=4096,K=4096,N=4096 |   PASS   |   PASS  
    M=4096,K=4096,N=6144 |   PASS   |   PASS  
   M=4096,K=4096,N=24576 |   PASS   |   PASS  
   M=4096,K=12288,N=4096 |   PASS   |   PASS  
INT32: 12/12 通过, BF16: 12/12 通过


### 5.2 测试算子性能
基于torch_npu.profiler来测试算子性能，执行下面的code cell测试单算子性能数据：

In [5]:
import os
import csv
from pathlib import Path

# ============================================================
# Profiler 配置
# ============================================================
PROF_OUTPUT_DIR = str(SRC_DIR / "prof_output")  # 采集数据的存放路径
WARMUP_ITERS = 0      # warmup 步数, 不采集数据

def parse_kernel_details(csv_path):
    """解析 kernel_details.csv, 提取 QmmCustom 每次执行的 Duration
    """
    rows = []
    with open(csv_path, newline="", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f)
        for row in reader:
            name = row.get("Name", "")
            if "QmmCustom" not in name:
                continue

            # 解析 Input Shapes: 格式如 "1,4096;4096,4096;4096;"
            raw_shapes = row.get("Input Shapes", "").strip('"')
            parts = [p.strip() for p in raw_shapes.split(";")]
            x1_shape = parts[0].split(",")  # (M, K)
            x2_shape = parts[1].split(",")  # (K, N)
            M = int(x1_shape[0])
            K = int(x1_shape[1])
            N = int(x2_shape[1])

            # 从 Output Data Types 判断 dtype
            out_dtype_raw = row.get("Output Data Types", "").strip('"')
            dtype_tag = "INT32" if "INT32" in out_dtype_raw else "BF16"

            # Duration(us)
            duration_us = float(row.get("Duration(us)", "0") or "0")

            rows.append({
                "M": M, "K": K, "N": N,
                "dtype": dtype_tag,
                "duration_us": duration_us,
            })
    return rows


def find_kernel_details_csv(prof_dir):
    prof_path = Path(prof_dir)
    if not prof_path.exists():
        return None
    # 找出所有包含 ASCEND_PROFILER_OUTPUT/kernel_details.csv 的子目录
    candidates = []
    for subdir in sorted(prof_path.iterdir(), key=lambda d: d.name, reverse=True):
        csv_path = subdir / "ASCEND_PROFILER_OUTPUT" / "kernel_details.csv"
        if csv_path.exists():
            candidates.append(csv_path)
            break  # sorted reverse=True, 第一个就是最新的
    return str(candidates[0]) if candidates else None

# ============================================================
# 主流程: 先 warmup, 再开启 profiler 采集, 最后汇总
# ============================================================
int32_results = []
bf16_results = []

# warmup: 不采集数据, 让 NPU 稳定
if WARMUP_ITERS>0:
    print("=" * 60)
    print(f"Warmup: 运行 {WARMUP_ITERS} 轮不采集数据")
    print("=" * 60)
    for i, (M, K, N) in enumerate(TEST_SHAPES, 1):
        test_int32(M, K, N)
        test_bf16(M, K, N)

# 创建 profiler
os.makedirs(PROF_OUTPUT_DIR, exist_ok=True)
profiler = torch_npu.profiler.profile(
    activities=[
        torch_npu.profiler.ProfilerActivity.NPU,
        torch_npu.profiler.ProfilerActivity.CPU,
    ],
    experimental_config=torch_npu.profiler._ExperimentalConfig(
        profiler_level=torch_npu.profiler.ProfilerLevel.Level1,
        aic_metrics=torch_npu.profiler.AiCMetrics.PipeUtilization,
    ),
    on_trace_ready=torch_npu.profiler.tensorboard_trace_handler(PROF_OUTPUT_DIR),
)

# 开启 profiler 采集
profiler.start()

for i, (M, K, N) in enumerate(TEST_SHAPES, 1):
    print("=" * 60)
    print(f"规格{i}: M={M}, K={K}, N={N}")
    print("=" * 60)

    print("[INT32 测试]")
    int32_pass = test_int32(M, K, N)
    int32_results.append((M, K, N, int32_pass))
    profiler.step()

    print("[BF16 测试]")
    bf16_pass = test_bf16(M, K, N)
    bf16_results.append((M, K, N, bf16_pass))
    profiler.step()

# 停止 profiler
profiler.stop()

# ============================================================
# 测试汇总
# ============================================================
print()
print("=" * 60)
print("测试汇总")
print("=" * 60)
print(f"{'规格':>24} | {'INT32':^8} | {'BF16':^8}")
print("-" * 46)
for (M, K, N, p1), (_, _, _, p2) in zip(int32_results, bf16_results):
    tag = f"M={M},K={K},N={N}"
    print(f"{tag:>24} | {'PASS' if p1 else 'FAIL':^8} | {'PASS' if p2 else 'FAIL':^8}")

int32_total = sum(p for _, _, _, p in int32_results)
bf16_total = sum(p for _, _, _, p in bf16_results)
total = len(TEST_SHAPES)
print(f"INT32: {int32_total}/{total} 通过, BF16: {bf16_total}/{total} 通过")

# ============================================================
# Profiler 数据汇总: 每种规格每种 dtype 的 Duration(us)
# ============================================================
print()
print("=" * 70)
print("Profiler 数据汇总 (QmmCustom Duration)")
print("=" * 70)

csv_path = find_kernel_details_csv(PROF_OUTPUT_DIR)
print(f"原始数据路径：{csv_path}")
if csv_path:
    kernel_rows = parse_kernel_details(csv_path)
    if kernel_rows:
        # 按 (M, K, N) 分组, 每组内有 INT32 和 BF16
        shape_map = {}
        for kr in kernel_rows:
            key = (kr["M"], kr["K"], kr["N"])
            shape_map.setdefault(key, {})[kr["dtype"]] = kr["duration_us"]

        print(f"{'规格 (M,K,N)':>24} | {'INT32 Duration(us)':>18} | {'BF16 Duration(us)':>18}")
        print("-" * 70)
        for M, K, N in TEST_SHAPES:
            key = (M, K, N)
            int32_dur = shape_map.get(key, {}).get("INT32", "-")
            bf16_dur = shape_map.get(key, {}).get("BF16", "-")
            int32_str = f"{int32_dur:.3f}" if isinstance(int32_dur, float) else int32_dur
            bf16_str = f"{bf16_dur:.3f}" if isinstance(bf16_dur, float) else bf16_dur
            tag = f"M={M},K={K},N={N}"
            print(f"{tag:>24} | {int32_str:>18} | {bf16_str:>18}")
    else:
        print(f"在 {csv_path} 中未找到 QmmCustom 算子数据")
else:
    print(f"未在 {PROF_OUTPUT_DIR} 中找到 kernel_details.csv 文件")
    print("请检查 profiler 输出目录")

print("=" * 70)


规格1: M=1, K=4096, N=4096
[INT32 测试]


  INT32 allclose验证: PASS
[BF16 测试]


  BF16 allclose验证: PASS
规格2: M=1, K=4096, N=6144
[INT32 测试]


  INT32 allclose验证: PASS
[BF16 测试]


  BF16 allclose验证: PASS
规格3: M=1, K=4096, N=24576
[INT32 测试]


...

/

-

[2026-09-10 18:53:22] [INFO] [1162707] profiler.py: CANN profiling data parsed in a total time of 0:00:02.019937

[2026-09-10 18:53:23] [INFO] [1162662] profiler.py: All profiling data parsed in a total time of 0:00:03.081103


测试汇总
                      规格 |  INT32   |   BF16  
----------------------------------------------
       M=1,K=4096,N=4096 |   PASS   |   PASS  
       M=1,K=4096,N=6144 |   PASS   |   PASS  
      M=1,K=4096,N=24576 |   PASS   |   PASS  
      M=1,K=12288,N=4096 |   PASS   |   PASS  
      M=50,K=4096,N=4096 |   PASS   |   PASS  
      M=50,K=4096,N=6144 |   PASS   |   PASS  
     M=50,K=4096,N=24576 |   PASS   |   PASS  
     M=50,K=12288,N=4096 |   PASS   |   PASS  
    M=4096,K=4096,N=4096 |   PASS   |   PASS  
    M=4096,K=4096,N=6144 |   PASS   |   PASS  
   M=4096,K=4096,N=24576 |   PASS   |   PASS  
   M=4096,K=12288,N=4096 |   PASS   |   PASS  
INT32: 12/12 通过, BF16: 12/12 通过

Profiler 数据汇总 (QmmCustom Duration)
原始数据路径：/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/prof_output/11b261c2a16148979a66e1bfd1575cdb_1153669_20260910185258896_ascend_pt/ASCEND_PROFILER_OUTPUT/kernel_details.csv
              规格 (M,K,N) | INT32 Duration(us)

---
## 6 自定义算子接入模型

### 6.1 定位替换点

Qwen3-8B 推理框架中的 W8A8 量化 matmul 通过 `CompressedTensorsW8A8Int8LinearMethod` 类实现，其 `apply_weights` 方法中使用 `torch_npu.npu_quant_matmul` 执行 INT8 量化矩阵乘法。我们需要将这个调用替换为自定义的 `torch.ops.ascendc_ops.qmm_custom` 算子。

查看原始实现：

In [6]:
!grep -n 'npu_quant_matmul' {RECIPE_ROOT}/module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py

93:        x = torch_npu.npu_quant_matmul(x, layer.weight,


### 6.2 替换实现

在 `CompressedTensorsW8A8Int8LinearMethod` 中，将 `torch_npu.npu_quant_matmul` 替换为 `torch.ops.ascendc_ops.qmm_custom`。

替换后的方法为：

```python
torch.ops.load_library(str(QMM_CUSTOM_DIR / "build/libascendc_ops.so"))

def apply_weights(self, layer: torch.nn.Module,
                x: torch.Tensor,
                bias: Optional[torch.Tensor],
                dynamic_scale: Optional = None,
                out_dtype: Optional = torch.bfloat16
                ) -> Union[torch.Tensor, Dict[str, Any]]:
    if dynamic_scale is not None:
        x_scale = dynamic_scale
    else:
        x, x_scale = torch_npu.npu_dynamic_quant(x, smooth_scales=layer.smooth_scales)
    out_shape_dim_0 = x.size()[:-1]
    x = x.view(-1, x.size(-1))
    x_scale = x_scale.view(-1)

    # --- 替换核心: npu_quant_matmul → qmm_custom ---
    x = torch.ops.ascendc_ops.qmm_custom(x, layer.weight, 
                                layer.weight_scale.view(-1),
                                pertoken_scale=None if out_dtype == torch.int32 else x_scale)
    # --- 替换结束 ---

    x = x.view(out_shape_dim_0 + (-1, ))
    if out_dtype == torch.int32:
        return x, x_scale
    else:
        return x
```

执行如下操作替换推理网络中调用的接口实现：

In [7]:
# 6.2 替换实现: 将 torch_npu.npu_quant_matmul 替换为 torch.ops.ascendc_ops.qmm_custom

# 1: 导入libascendc_ops.so
W8A8_FILE = RECIPE_ROOT / 'module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py'
print('[替换] 目标文件:', W8A8_FILE)

# 备份原始内容，仅在首次执行时保存，避免重复执行时覆盖为已修改的代码
if 'W8A8_ORIGINAL_CODE' not in dir():
    W8A8_ORIGINAL_CODE = W8A8_FILE.read_text(encoding='utf-8')

ct_code = W8A8_FILE.read_text(encoding='utf-8')
_lib_path = str(QMM_CUSTOM_DIR / "build/libascendc_ops.so")
_load_lib_stmt = 'torch.ops.load_library(r"' + _lib_path + '")'
if _load_lib_stmt not in ct_code:
    # 在 'import torch' 行后插入 'torch.ops.load_library'
    ct_code = ct_code.replace('import torch\n', 'import torch\n' + _load_lib_stmt + '\n')
    W8A8_FILE.write_text(ct_code, encoding='utf-8')
    print('[插入] compressed_tensors.py: 已添加 torch.ops.load_library')
else:
    print('[插入] compressed_tensors.py: torch.ops.load_library 已存在，跳过')

# 2: 在 compressed_tensors_w8a8_int8.py 中替换 npu_quant_matmul → qmm_custom
code = W8A8_FILE.read_text(encoding='utf-8')

# 替换 npu_quant_matmul 调用为 qmm_custom 调用
old_call = (
    'x = torch_npu.npu_quant_matmul(x, layer.weight,\n'
    '                                    layer.weight_scale.view(-1),\n'
    '                                    pertoken_scale=None if out_dtype == torch.int32 else x_scale,\n'
    '                                    bias=layer.bias,\n'
    '                                    output_dtype=out_dtype)'
)
new_call = (
    '# --- 替换核心: npu_quant_matmul → qmm_custom ---\n'
    '        x = torch.ops.ascendc_ops.qmm_custom(x, layer.weight,\n'
    '                                    layer.weight_scale.view(-1),\n'
    '                                    pertoken_scale=None if out_dtype == torch.int32 else x_scale)\n'
    '        # --- 替换结束 ---'
)

if old_call in code:
    code = code.replace(old_call, new_call)
    W8A8_FILE.write_text(code, encoding='utf-8')
    print('[替换] 成功: npu_quant_matmul → qmm_custom')
else:
    print('[替换] 文件中未找到 npu_quant_matmul 调用，可能已替换')

# 验证替换结果
result = W8A8_FILE.read_text(encoding='utf-8')
if 'torch.ops.ascendc_ops.qmm_custom' in result:
    print('[验证] 替换已生效，当前使用 qmm_custom 算子')
else:
    print('[验证] 替换未生效，请检查文件内容')


[替换] 目标文件: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py
[插入] compressed_tensors.py: 已添加 torch.ops.load_library
[替换] 成功: npu_quant_matmul → qmm_custom
[验证] 替换已生效，当前使用 qmm_custom 算子


### 6.3 替换实现后执行量化推理

指定本地的量化模型权重路径，调用infer.sh执行量化推理，查看 W8A8 量化模型推理生成的文本，对比替换前的生成文本是否一致。

**注意：需要先完成第五章的量化模型推理过程，并将`W8A8_WEIGHT`指定为本地的量化模型权重的路径**

In [8]:
import sys, subprocess, os
from inference_scripts.recipe_workflow import prepare_runtime_yaml

# 将W8A8_WEIGHT设置为量化模型权重实际所在路径
W8A8_WEIGHT = TUTORIAL_DIR / 'src/data/models/Qwen3-8B-W8A8'

YAML_A8W8_TEMPLATE =  RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_1tp.yaml'
YAML_A8W8_CUSTOM  = RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_custom.yaml'
prepare_runtime_yaml(YAML_A8W8_TEMPLATE, YAML_A8W8_CUSTOM, model_path=str(W8A8_WEIGHT))

env = os.environ.copy()
python_dir = os.path.dirname(sys.executable)
env["PATH"] = python_dir + ":" + env.get("PATH", "")
result = subprocess.run(
    ['bash', 'executor/scripts/infer.sh', '--model', 'qwen', '--yaml', YAML_A8W8_CUSTOM],
    cwd=str(RECIPE_ROOT),
    env=env,
    check=True,
)

rm: cannot remove '/root/atc_data/': Permission denied


====================> 启动离线推理 (model=qwen, yaml=/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/config/qwen3_8b_a8w8_custom.yaml)
world_size = 1


[INFO][2026-09-10 18:53:31.679][patch_getenv.py:15] get env LOCAL_RANK = 0
[INFO][2026-09-10 18:53:31.679][patch_getenv.py:15] get env RANK_OFFSET = 0
[INFO][2026-09-10 18:53:31.680][patch_getenv.py:15] get env LC_ALL = en_US.UTF-8
[INFO][2026-09-10 18:53:31.682][patch_getenv.py:15] get env WORK_DIR = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen
[INFO][2026-09-10 18:53:31.682][patch_getenv.py:15] get env RES_PATH = res/20260910/qwen3_8b_qwen3_8b_a8w8_custom
[INFO][2026-09-10 18:53:31.682][infer.py:146] Inference Configuration
[INFO][2026-09-10 18:53:31.682][infer.py:147] InferenceConfig(model_config=ModelConfig(model_name='qwen3_8b', model_path='/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/data/models/Qwen3-8B-W8A8', output_path='/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260

[INFO][2026-09-10 18:53:32.679][execution_engine.py:126] Loading main model...
/mnt/workspace/.venv/lib/python3.11/site-packages/transformers/modeling_rope_utils.py:1036: FutureWarning: `rope_config_validation` is deprecated and has been removed. Its functionality has been moved to RotaryEmbeddingConfigMixin.validate_rope method. PreTrainedConfig inherits this class, so please call self.validate_rope() instead. Also, make sure to use the new rope_parameters syntax. You can call self.standardize_rope_params() in the meantime.
  warnings.warn(
[INFO][2026-09-10 18:53:32.767][model_worker.py:185] Try to load pretrained model in path: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/data/models/Qwen3-8B-W8A8
[INFO][2026-09-10 18:53:32.767][comm_manager.py:442] CommManager: group 'attn_tp_group' registered (my_subgroup=[0], reuse=True, rank_in_group=0)
[INFO][2026-09-10 18:53:32.767][comm_manager.py:442] CommManager: group 'lmhead_tp_group' registered (my_s

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading safetensors checkpoint shards:   0% Completed | 0/37 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  11% Completed | 4/37 [00:00<00:00, 36.19it/s]


...

[INFO][2026-09-10 18:54:09.231][scheduler.py:518] [Main] Inference time (decode): 39.99 ms
[INFO][2026-09-10 18:54:09.272][scheduler.py:518] [Main] Inference time (decode): 39.69 ms
[INFO][2026-09-10 18:54:09.327][scheduler.py:518] [Main] Inference time (decode): 53.46 ms
[INFO][2026-09-10 18:54:09.373][scheduler.py:518] [Main] Inference time (decode): 44.32 ms
[INFO][2026-09-10 18:54:09.420][scheduler.py:518] [Main] Inference time (decode): 44.44 ms


[INFO][2026-09-10 18:54:09.466][scheduler.py:518] [Main] Inference time (decode): 44.14 ms
[INFO][2026-09-10 18:54:09.511][scheduler.py:518] [Main] Inference time (decode): 43.49 ms
[INFO][2026-09-10 18:54:09.580][scheduler.py:518] [Main] Inference time (decode): 67.88 ms
[INFO][2026-09-10 18:54:09.625][scheduler.py:518] [Main] Inference time (decode): 42.98 ms


[INFO][2026-09-10 18:54:09.669][scheduler.py:518] [Main] Inference time (decode): 42.85 ms
[INFO][2026-09-10 18:54:09.714][scheduler.py:518] [Main] Inference time (decode): 43.40 ms
[INFO][2026-09-10 18:54:09.759][scheduler.py:518] [Main] Inference time (decode): 43.14 ms
[INFO][2026-09-10 18:54:09.803][scheduler.py:518] [Main] Inference time (decode): 42.71 ms
[INFO][2026-09-10 18:54:09.848][scheduler.py:518] [Main] Inference time (decode): 42.95 ms


[INFO][2026-09-10 18:54:09.892][scheduler.py:518] [Main] Inference time (decode): 43.15 ms
[INFO][2026-09-10 18:54:09.936][scheduler.py:518] [Main] Inference time (decode): 42.43 ms
[INFO][2026-09-10 18:54:09.980][scheduler.py:518] [Main] Inference time (decode): 42.46 ms
[INFO][2026-09-10 18:54:10.024][scheduler.py:518] [Main] Inference time (decode): 42.16 ms
[INFO][2026-09-10 18:54:10.068][scheduler.py:518] [Main] Inference time (decode): 42.24 ms


[INFO][2026-09-10 18:54:10.111][scheduler.py:518] [Main] Inference time (decode): 42.26 ms
[INFO][2026-09-10 18:54:10.112][infer.py:99] Request 0: outputs: The output of an **attention function** is typically a **weighted sum** of the **value vectors**, where the weights are determined by the **similarity** between the **query vector** and the **key vectors** from the set of key-value pairs.

### Formal Description:

Given:
- A **query vector** $ Q $
- A set of **key-value pairs**: $ (K_1, V_1), (K_2, V_2), \dots, (K_n, V_n) $

The **attention function** computes the output as:

$$
\text{Attention}(Q, K, V) = \sum_{i=1}^{n} \alpha_i V_i
$$

Where:
- $ \alpha_i = \text{softmax}\left( \frac{Q \cdot K_i}{\sqrt{d_k}} \right) $
- $ d_k $ is the dimensionality of the key vectors (used for scaling to prevent large values)
- $ \cdot $ denotes the dot product
- $ \text{softmax} $ ensures that the weights $ \alpha_i $ sum to 1 and represent attention distribution

### Final Output:
The **output*

---
## 7 模型性能测试

### 任务六：测试模型性能，尝试对自定义算子进行优化后重复上述流程

将 QmmCustom 算子接入模型后，使用 Profiling 工具观测模型推理性能，对比自定义算子替换前后的 matmul 耗时变化。

### 7.1 准备 Profiling YAML

沿用 recipes 的 YAML + `infer.sh` 工作流，在配置中打开 Profiler，运行一次推理并收集性能数据。

通过如下操作将 YAML 配置中的enable_profiler配置为 True ：

In [9]:
from inference_scripts.recipe_workflow import prepare_profiling_yaml

YAML_A8W8_CUSTOM_PROFILER = RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_custom_profiler.yaml'

prepare_profiling_yaml(YAML_A8W8_CUSTOM, YAML_A8W8_CUSTOM_PROFILER, enable_profiler=True)

{'model_config': {'model_name': 'qwen3_8b',
  'model_path': '/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/data/models/Qwen3-8B-W8A8',
  'exe_mode': 'eager',
  'enable_profiler': True,
  'with_ckpt': True,
  'enable_cache_compile': False,
  'enable_static_kernel': False},
 'data_config': {'dataset': 'default', 'input_truncated_len': 4096},
 'parallel_config': {'world_size': 1,
  'attn_tp_size': 1,
  'moe_tp_size': 1,
  'embed_tp_size': 1,
  'lmhead_tp_size': 1},
 'scheduler_config': {'max_new_tokens': 256, 'batch_size': 1}}

### 7.2 启动 Profiling 推理

使用 recipes 命令启动带 Profiler 的推理，推理结束后会在结果目录中生成 `prof/` 产物。

In [10]:
result = subprocess.run(
    ['bash', 'executor/scripts/infer.sh', '--model', 'qwen', '--yaml', YAML_A8W8_CUSTOM_PROFILER],
    cwd=str(RECIPE_ROOT),
    env=env,
    check=True,
)

rm: cannot remove '/root/atc_data/': Permission denied


====================> 启动离线推理 (model=qwen, yaml=/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/config/qwen3_8b_a8w8_custom_profiler.yaml)
world_size = 1


[INFO][2026-09-10 18:54:25.980][patch_getenv.py:15] get env LOCAL_RANK = 0
[INFO][2026-09-10 18:54:25.980][patch_getenv.py:15] get env RANK_OFFSET = 0
[INFO][2026-09-10 18:54:25.980][patch_getenv.py:15] get env LC_ALL = en_US.UTF-8
[INFO][2026-09-10 18:54:25.982][patch_getenv.py:15] get env WORK_DIR = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen
[INFO][2026-09-10 18:54:25.982][patch_getenv.py:15] get env RES_PATH = res/20260910/qwen3_8b_qwen3_8b_a8w8_custom_profiler
[INFO][2026-09-10 18:54:25.982][infer.py:146] Inference Configuration
[INFO][2026-09-10 18:54:25.982][infer.py:147] InferenceConfig(model_config=ModelConfig(model_name='qwen3_8b', model_path='/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/data/models/Qwen3-8B-W8A8', output_path='/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/

[INFO][2026-09-10 18:54:26.991][execution_engine.py:126] Loading main model...
/mnt/workspace/.venv/lib/python3.11/site-packages/transformers/modeling_rope_utils.py:1036: FutureWarning: `rope_config_validation` is deprecated and has been removed. Its functionality has been moved to RotaryEmbeddingConfigMixin.validate_rope method. PreTrainedConfig inherits this class, so please call self.validate_rope() instead. Also, make sure to use the new rope_parameters syntax. You can call self.standardize_rope_params() in the meantime.
  warnings.warn(
[INFO][2026-09-10 18:54:27.077][model_worker.py:185] Try to load pretrained model in path: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/data/models/Qwen3-8B-W8A8
[INFO][2026-09-10 18:54:27.078][comm_manager.py:442] CommManager: group 'attn_tp_group' registered (my_subgroup=[0], reuse=True, rank_in_group=0)
[INFO][2026-09-10 18:54:27.078][comm_manager.py:442] CommManager: group 'lmhead_tp_group' registered (my_s

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading safetensors checkpoint shards:   0% Completed | 0/37 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  11% Completed | 4/37 [00:00<00:00, 37.77it/s]


...

[INFO][2026-09-10 18:55:13.189][scheduler.py:518] [Main] Inference time (decode): 39.27 ms
[INFO][2026-09-10 18:55:13.287][scheduler.py:518] [Main] Inference time (decode): 97.01 ms
[INFO][2026-09-10 18:55:13.350][scheduler.py:518] [Main] Inference time (decode): 61.21 ms


[INFO][2026-09-10 18:55:13.401][scheduler.py:518] [Main] Inference time (decode): 49.25 ms
[INFO][2026-09-10 18:55:13.469][scheduler.py:518] [Main] Inference time (decode): 66.03 ms
[INFO][2026-09-10 18:55:13.542][scheduler.py:518] [Main] Inference time (decode): 71.66 ms
[INFO][2026-09-10 18:55:13.592][scheduler.py:518] [Main] Inference time (decode): 48.65 ms


[INFO][2026-09-10 18:55:13.641][scheduler.py:518] [Main] Inference time (decode): 46.57 ms
[INFO][2026-09-10 18:55:13.731][scheduler.py:518] [Main] Inference time (decode): 88.60 ms
[INFO][2026-09-10 18:55:13.782][scheduler.py:518] [Main] Inference time (decode): 49.05 ms
[INFO][2026-09-10 18:55:13.837][scheduler.py:518] [Main] Inference time (decode): 53.66 ms


[INFO][2026-09-10 18:55:13.883][scheduler.py:518] [Main] Inference time (decode): 44.72 ms
[INFO][2026-09-10 18:55:13.934][scheduler.py:518] [Main] Inference time (decode): 48.86 ms
[INFO][2026-09-10 18:55:13.980][scheduler.py:518] [Main] Inference time (decode): 44.90 ms
[INFO][2026-09-10 18:55:14.038][scheduler.py:518] [Main] Inference time (decode): 56.28 ms


[INFO][2026-09-10 18:55:14.084][scheduler.py:518] [Main] Inference time (decode): 44.53 ms
[INFO][2026-09-10 18:55:14.134][scheduler.py:518] [Main] Inference time (decode): 48.38 ms
[INFO][2026-09-10 18:55:14.182][scheduler.py:518] [Main] Inference time (decode): 46.05 ms
[INFO][2026-09-10 18:55:14.236][scheduler.py:518] [Main] Inference time (decode): 52.19 ms
[INFO][2026-09-10 18:55:14.236][infer.py:99] Request 0: outputs: The output of an **attention function** is typically a **weighted sum** of the **value vectors**, where the weights are determined by the **similarity** between the **query vector** and the **key vectors** from the set of key-value pairs.

### Formal Description:

Given:
- A **query vector** $ Q $
- A set of **key-value pairs**: $ (K_1, V_1), (K_2, V_2), \dots, (K_n, V_n) $

The **attention function** computes the output as:

$$
\text{Attention}(Q, K, V) = \sum_{i=1}^{n} \alpha_i V_i
$$

Where:
- $ \alpha_i = \text{softmax}\left( \frac{Q \cdot K_i}{\sqrt{d_k}} \rig

### 7.3 查看 Profiling 结果

推理完成后，定位 `prof/` 目录，重点关注其中的 `op_statistic.csv` 和 `kernel_details.csv` 这两个文件。`op_statistic.csv`中展示了模型调用的所有算子的统计信息，可快速确认自定义算子 QmmCustom 的调用情况。`kernel_details.csv`中展示了详细的接口调用信息，可自行查看筛选自己关注的算子的性能。

通过如下操作统计 `kernel_details.csv` 中的 QmmCustom 算子耗时，可自行对比自定义算子替换前后的性能差异。

In [11]:
import pandas as pd
from pathlib import Path

RES_DIR = RECIPE_ROOT / 'models/qwen/res'

# 取最新的 profiling 结果目录
result_dirs = sorted([d for d in RES_DIR.iterdir() if d.is_dir()], key=lambda d: d.name, reverse=True)
latest_result = result_dirs[0]
print(f'最新结果目录: {latest_result}')

def find_kernel_csv(phase_dir):
    matches = sorted(phase_dir.rglob('kernel_details.csv'), key=lambda p: p.parent.parent.name, reverse=True)
    return matches[0] if matches else None

prefill_csv = find_kernel_csv(latest_result / 'qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/prefill')
decode_csv = find_kernel_csv(latest_result / 'qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/decode')
assert prefill_csv and decode_csv, 'kernel_details.csv 未找到，请确认 profiling 已完成'
print(f'Prefill CSV: {prefill_csv}')
print(f'Decode CSV: {decode_csv}')

# 读取 CSV 并筛选 QmmCustom 算子
prefill_df = pd.read_csv(prefill_csv)
decode_df = pd.read_csv(decode_csv)

prefill_qmm = prefill_df[prefill_df['Type'].str.contains('QmmCustom')].copy()
decode_qmm = decode_df[decode_df['Type'].str.contains('QmmCustom')].copy()

# 按 Input Shapes 分组，统计每种 shape 下的平均 Duration
def qmm_duration_stats(df, phase):
    stats = df.groupby(['Input Shapes', 'Output Data Types'])['Duration(us)'].agg(['mean', 'count']).reset_index()
    stats.columns = ['Input Shapes', 'Output Dtype', 'Avg Duration(us)', 'Calls']
    stats = stats.sort_values('Avg Duration(us)', ascending=False).reset_index(drop=True)
    stats.insert(0, 'Phase', phase)
    return stats

prefill_stats = qmm_duration_stats(prefill_qmm, 'Prefill')
decode_stats = qmm_duration_stats(decode_qmm, 'Decode')

combined = pd.concat([prefill_stats, decode_stats], ignore_index=True)
display(combined.style.set_caption('QmmCustom 各 Shape 平均耗时 (us)'))

最新结果目录: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260910
Prefill CSV: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260910/qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/prefill/11b261c2a16148979a66e1bfd1575cdb_1166883_20260910185452631_ascend_pt/ASCEND_PROFILER_OUTPUT/kernel_details.csv
Decode CSV: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260910/qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/decode/11b261c2a16148979a66e1bfd1575cdb_1166883_20260910185457484_ascend_pt/ASCEND_PROFILER_OUTPUT/kernel_details.csv


,Phase,Input Shapes,Output Dtype,Avg Duration(us),Calls
0,Prefill,"""50,4096;4096,24576;24576""",INT32,101.071111,36
1,Prefill,"""50,12288;12288,4096;4096;50""",DT_BF16,62.782222,36
2,Prefill,"""50,4096;4096,6144;6144;50""",DT_BF16,39.297222,36
3,Prefill,"""50,4096;4096,4096;4096;50""",DT_BF16,28.171111,36
4,Decode,"""1,4096;4096,24576;24576""",INT32,81.728333,108
5,Decode,"""1,12288;12288,4096;4096;1""",DT_BF16,50.075185,108
6,Decode,"""1,4096;4096,6144;6144;1""",DT_BF16,33.007037,108
7,Decode,"""1,4096;4096,4096;4096;1""",DT_BF16,23.275370,108


### 7.4 恢复原始 W8A8 源码

自定义算子推理与 Profiling 已完成，下方单元格会从备份恢复 `compressed_tensors_w8a8_int8.py` 原始内容，避免修改后的源码污染 baseline 对比或产生 git 工作区变更。

In [12]:
W8A8_FILE = RECIPE_ROOT / 'module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py'
W8A8_FILE.write_text(W8A8_ORIGINAL_CODE, encoding='utf-8')
print('[恢复] 已恢复原始 W8A8 源码，baseline 对比不受污染')

[恢复] 已恢复原始 W8A8 源码，baseline 对比不受污染


## 8 提交要求

> 完整的提交说明参考[启航营课程实践说明](https://gitcode.com/cann/cann-launch-camp/blob/master/2026/University/HIT/First-session/README.md)。

1. 提交完整的 Jupyter Notebook（.ipynb），**保留所有单元格输出**。
2. 提交前依次运行所有单元格，确保：
   - 自定义算子编译成功；
   - 算子功能测试通过；
   - 性能数据收集成功。
3. **文件命名**：`gitcode账号_result.ipynb`。
4. **允许并且鼓励**使用 AI 工具辅助，但须独立思考并在报告中简述（解决了什么疑惑、是否引入新 bug）。
5. **不得抄袭。**

---
### 小结

本章完成了 QmmCustom 算子的开发、编译和单算子功能性能测试，并将该自定义算子接入了 Qwen3-8B 量化模型，验证功能并观测了其在模型中的性能。


### 附录
实践相关参考材料链接：
- [Ascend C 高阶api文档](https://gitcode.com/cann/asc-devkit/blob/master/docs/api/SIMD-API/%E9%AB%98%E9%98%B6API/%E9%AB%98%E9%98%B6API.md)
- [昇腾社区文档](https://www.hiascend.com/zh/document)
- [Ascend C 算子开发系列教程](https://gitcode.com/cann/cann-learning-hub/tree/master/tutorials/ascendc_operator_development)
- [Ascend C 算子功能调试](https://gitcode.com/cann/cann-learning-hub/tree/master/tutorials/ascendc_operator_development/07_Troubleshooting)
- [Matmul最佳实践样例](https://gitcode.com/cann/asc-devkit/blob/master/examples/01_simd_cpp_api/05_best_practices/01_matrix_compute/matmul_high_performance/README.md)
- [CANN内置量化矩阵乘算子实现](https://gitcode.com/cann/ops-nn/tree/master/matmul/quant_batch_matmul_v3)